# AP Model 3D Foto MAX: usa toda la GPU y la RAM (A100 / L4 / T4)

Foto → modelo 3D con textura HD, **esqueleto automático compatible con Mixamo**,
**pesos sin huecos**, **6 animaciones**, piezas por hueso y exportación a
**Blender (.blend) / FBX / GLB animado / OBJ / STL / video MP4**.

**Qué hace distinto esta versión**
* Se ajusta sola al equipo. Con una **A100 de 80 GB + 167 GB de RAM** usa el perfil **MÁXIMO**: forma octree 512, hasta 150 000 caras (antes ~25 000), textura de 4096 px y video de 1080p.
* **Usa la RAM a fondo**: carga en la RAM los pesos de los modelos (cerca de 30 GB) y deja **todos los modelos cargados a la vez** en la GPU (1 foto, varias vistas y textura). Trabaja en `/dev/shm`, un disco en RAM, guarda en caché los resultados y deja que Blender corra **en paralelo** con la siguiente generación.
* **Pesos sin manchas**: los calcula Blender (método de calor) sobre una copia cerrada de la malla y los pasa al modelo real. Cada hueso mueve su zona completa, no solo una banda.
* **Turbo**: forma con 5 pasos + FlashVDM, textura turbo y Blender en dos procesos a la vez (archivos y videos). Barra con **porcentaje y tiempo restante** y **monitor en vivo** de GPU/VRAM/RAM/CPU/temperatura que se pone **en rojo** si algo se satura o hay cola.
* **Ves el modelo antes**: el estudio se abre en cuanto está el GLB animado, mientras los videos siguen renderizándose. El visor usa un **GLB ligero** (Draco + JPEG) que carga mucho más rápido en el celular. La GPU se **precalienta** al cargar y los tiempos aprendidos se guardan entre sesiones.
* **Esqueleto detallado (66 huesos)**: dedos de las manos (5 × 3 falanges, nombres de Mixamo), dedos de los pies, ojos y mandíbula, colocados midiendo la malla. Animaciones nuevas **Manos** y **Hablar**; los dedos se relajan o cierran en las demás.
* **Por etapas, de más a menos**: primero la malla completa de la IA en alta resolución (`malla_alta.glb`, pestaña *Alta*), luego se reduce a las caras pedidas para la textura y el esqueleto.
* **Física de telas**: túnicas, capas, mangas y faldas se liberan de piernas y brazos y cuelgan de la cadera o la espalda, para que no se rasguen al animar. Los vértices duplicados se sueldan y los pesos se calculan con hasta 3 intentos.
* **Huesos de tela con movimiento secundario**: en túnicas, faldas y capas se crean 4 cadenas de huesos (frente, atrás, izquierda, derecha) que se balancean con retraso e inercia en cada animación. Queda grabado en GLB, FBX y .blend.
* **Piezas con nombre**: Cabeza, Torso, Brazo izq., Mano der., Dedos pie izq.…
* **Malla en cuadrados limpia** (como *Smart Mesh* de Tripo): retopología QuadriFlow (o rejilla de voxeles si falla), UV nuevas y **color + relieve horneados** desde la malla original para no perder detalle. Con esqueleto y animaciones (`modelo_quads.glb`).
* **Detecta qué es**: persona, animal de 4 patas, **cabeza/busto** (mandíbula, ojos y cuello: Hablar, Mirar, Asentir, Negar, Sorpresa) u **objeto** (Flotar, Girar, Rebotar, Bailar, Saludar).
* **Texto a 3D** (como Tripo): escribe una descripción y se crea la imagen con HunyuanDiT y luego el modelo.
* **Textura rápida** (opción en Calidad, activa con Turbo): hornea a 1024 en lugar de 2048.
* **Dos motores de forma**: Hunyuan3D-2 (Turbo, segundos) o **TripoSG** del equipo de Tripo AI (más detalle). Blender **4.5 LTS**.
* **Malla reparada**: quita trozos sueltos, cierra agujeros y orienta las caras.
* **Cero vértices sin peso**: si alguno queda suelto, copia el peso de su vecino; el informe dice cuántos vértices mueve cada hueso.
* **LOD para juegos**: además del modelo completo, versiones al 50 % y 15 % con el mismo esqueleto y animaciones.
* **Visor web** (`visor.html` en el ZIP): botones para cada animación, cambio de calidad y modo AR en el celular.
* **Multivista**: si subes espalda o perfiles (o una hoja de vistas), usa Hunyuan3D-2mv y pinta también la espalda.

## Cómo usarlo
1. *Entorno de ejecución › Cambiar tipo de entorno de ejecución* → **GPU (A100 recomendado)** y **RAM alta**.
2. Ejecuta las celdas **1, 2 y 3** en orden (la primera vez tardan unos minutos).
3. **Opcional pero recomendado:** ejecuta la **3b**. Genera un modelo de ejemplo sin página web y muestra los tiempos. Si esa celda termina con "LISTO", el motor funciona.
4. Abre la **5. Página web** (o la 4 dentro del notebook). Verás tres pestañas que se llenan en orden: **1 · Malla → 2 · Textura → 3 · Estudio animado**.
5. Para no perder tus modelos, usa la **6. Guardar en Drive**.

**Para actualizar a un notebook nuevo sin reiniciar:** ejecuta la celda **1** (cambia el código y conserva los modelos cargados) y después la **5**.

## 1. Proyecto (código incluido: no necesita Drive ni subir nada)

In [ ]:
# @title 1. Proyecto (código incluido: no necesita Drive ni subir nada)
import base64, json, os, subprocess, sys, zlib
APP = "/content/ap3d_app"
os.makedirs(f"{APP}/ap3d", exist_ok=True)
_FILES = json.loads(r"""{"__init__.py": "eNoDAAAAAAE=", "config.py": "eNq9Vttu4zYQfddXEMpD5F1Bli9xXBcq4FyaBkiaIE2AAkEgMBJtayOJXJLyJi32u/reL+sMScXybpLdfakAWxQ5PDNnZjgc3/ePmGaZpqSk5OTyhjzh4Gp+DgNWFksG/0QwuShKUlFFaKk5+dgwktF7FnneIa9JU1MyH8QxyRmZxuTkAPYOJvs4gBnEUow0inagzud/np5fzDzNc65ICb+K5wzfIArwOa1Bg1xSXGZ1a5yxcs3+Cs0WwRRXHmwQkhnh2omiyiCj2YqhAYIuixpMf0/6OVv31arqWZYVLUsKUNTT7FE3khrOZF3kjBNFSwAzCiXoKZusAKrOBTTyfN/3ikpwqQkY4UZq1eii9NpPoezn1cXFNUlALmL1upC8jpZMB/78cnSU4pofEr+f8VqzWvepGOV+z7u4uU6PTq9e2wbLX+1KweYipwp2H5wd/350/Oput7yFcA98cybbN4B4Xs4WJF2KJujNPAKPlk92gI8jqbnMVpvJhZ2IsianUaFSuqZFSQG0xWgfAcZ1RMG4FMJTZCwVkkOa6IKpIO5t7ZEMwlQTEdW0YiG8Nde0TCtWcflE+mT47t0oNjvYY8aEJsfmBZHb6BZUKa8D5h9e3oAj4ih2hCWt0rxQD0FV1OlCMpYu75Opsx7CfkilYHBgILW0pPf0A8e0Mzm3STBVkBWFdFKCQuL8jN81DwngZjzC3PnKnagJXGJTKEIDUjgzSxb4Larfi4xQl6fzuZn/JSEdk7e9nQP0M5DNFme8vyUH6VLRB5YXUgV5CG4slE75Q3ItG/ZiLEAIFX6v03ObkoLqVfSBF3WA+Q9p2BpjdXzbio32TeByU8naPLMpsoZogkqbxHanmbBHM4KDoZvnDAp6NqG2PJyJRlmjYZRmvKlBBeGSDM3yJy4fQohalboAPqcPHCAXHWMERGc/nll37Nh6aYtln/wGH173XOQF8LClMvFtrYQU5ZkGFcneYAgKNRMq2YtDkq2a+kEl4xifcCtENq9oxlQy2DPLBCodiP40CcGDeMyTYTyehlDTypTX6CXj4xdg+GJRcponv9JSoV+xSKYSkeMp4CpaiRK+huPQuIRJlYysw+Ea2bhgNJ11HDBGB7zOfH523eE9RkO/4j38Bu/h/897+ALvYex4n43JcPwm6/Pjo9MO7dF03NIeb2hP3mQ93ZC2PF8jbVn9AOv9YYf0YPICacVmX4DtkOsxGey9Sfpg/sfpYYf1cG/Ssh5tWE/fIj1+hfQgxuD8AGmTCd/DeeCO+A7BNqhqshVt251lQ6VpYrgooGeAu6Lb5MB9AbcOVEXoOWruMNa8XDOJLQdjEqVA1l4Y0Jdgl8NkVcBVbZoQVy8N8oG9sR0MCCBuCYpI4JZM94Ua36N9IegEwyoBaPBuu6teZBDErY8lzLRPqTFY+Xe2rmEiT+JWCroudFf6iRXLld4SGsdt6XMTcAw7l8Gt7zxo9lT0MehOhcQVDhE1IqeaBRi0TTnH61hCGc4D/ArJoBeSrel2djvCWMcT/LPxgxotk+3ivUFoZxBm674Rz1eNymRxzwLhbhu3Hix87FP/Fre7YPTu3WcS4NiZjd8nBz3y7z8mTXClu4Dz27fxwkcZNBolMIAg4lpoXLFDXIN5e3LMvB26+S8hV1RB94Ji5tzs3s3Cz9hsQ1aBdNsK4zqMEUM8vgTT6X1QtvUpbIBL/D/sVZwi", "engine.py": "eNrNfW1z2zaX6Hf9CiwznZKtzEh2kqa6VWeT2kmz10l8k7R99nE9HEqEJMYUyZKUbCWT/Vv7fX/ZPS8ACb7JSprdWU9i8wU4AA7OOw5Ay7JeJkWSTcQC/ti5I45+Fms/inxxcirsXzfxbuPHJ6dHx+K+OF5v6X0hb4tN5uPl00jGgczcweC3PBGBFJmcb7I8yScDIb4T75IgyUUE/9dJIPHvXxsZ+LGY+9nSx3cyFtDW84vfhI8XW/lB2KnMFmEkXj75x4uXr8V9gCTEk/N3r52h8PNQxIlY+TshcyjnI4Qik2IT+zQEsUMwebjchPBCutSNc2goldArkUuRQhex9Vi1/ebJS2HP/flKYv9TfxnGABUuz8N4c+uISO5gjAn1AkfgZ/NVuOWup37mRzCuiVBAM+iIGqso/CyArsjlBqrn3JOzSBSZP/Pfw2u5ThOoLrY+grofyO39fLUWdhDm8wQfQc8cNQBGEfU+KeRSinkSY1NzPw4Aj/9HT4Swf4Fy31NVKJMBZmRMXdddpZomisRSxoDIeZjE1WxUeMtkvokKmitofrnBQVFBxDijjbsqYDQwMfkKcRfp2ajQ9n6TFzJHTKWyCDMssg7ztSoocxHGeeHH8E8m7sCyrEGICCqww0BTGfTVXWyA8KAoTNB8od8v5+VVlMz0NfYkCsvbMNFX7/Mk1tdJrq8yqa/y1aYIo/JuVxbJNzPA/lzm5ZNilUk/CONl+SBcl3A+hClQsRzo+3izTnfY8zgdLLJkLS5enAv17sXaX8oh/3md5gMu4IoKA4twORj8evbmTEyh127qFys3CLPYX0tb3/uzHP/anocNe57jDGAqvfPXv/xfqFV21j1P5te2M/j9xdkfb+GFbUFrcWENhTXz59f4N5ILus/C5aqwnMFgcE+8BULOQqC2CXTLR6LJxTIEPoJZFYlY+FGOg0tECiwOJFFs1jPgBxkB3WTbMEiyAY3PBb72Xrx88vzMu3jxj7Nz7MKjB95oNML/A3z79sUpjvN49OAxNB3Ihcj9hfSoWTtcL50JkRXQyO9+BF0Cpo+TDMRW+MEnuuQO4ozhW7vw136cDJEx4fcCixaJ4yKNIZxwAdWBRHKmwDm1oWbDpd+qQfzJ/BD4ANrdyLMsSzLbAp5WMgFHD8gwerCl/gEGserNUKxgWADczcMPUrd9I34SJ8cCuYeu9rV1XkKWKKPWIA+BNUG4gVyFVu11GIfrBMGlt45qlpr4DoD/LLon4AsaXGYgeGTVwNq/tXF4DjSiZ7ACCyNW454n6c52zBdusQJCif0wsm1dc1jCcPQ0nD959cs/X78tG8SaKGd55mIg4jfPnyLJwp8nltPddryVWWGrIlQikyBQYnytCG0eST/epF6yKdJNkdvXUqbTxyOgnTD2FpmU3nI2HY8qCnwKQtYnCWdISkCT2IbyPQrNkLRViTrFK4UvSM6XRFhku6rTQQ59zoH1ZWAHSLAiwEGifHPxV8ny75Mwtlk8uK9/e+edvngDOPjOchxEki4V5iAq7MCpEI8jwSZI1oEgya+9TY7cVYfluFQQVP93352Myto3K5Av2EkgAmGD5rGDHGcekYV0TJV+MlFmTAiND9oOoHNJao+c2hvVo2wNSl3awVCES2Bt6Ukkx3z6LtvIegVjmPI2zGHKAtB/lgvi12q0ij9QNpPrZCvNYrVSX44aeTuXaSHO6A+o06r11AelwQT2F2jdwouSZW5XRPR6jpQj/G2Ys42EkgRE3hzEkQAToSQtkGUJyBoX2A8VBsBNkC0JThAuNjk0Cyzz6tlr1MKr3UkArOuQ8iewEmEtAU1ZItYbmaPplCdgFcALf4uyr5KKrHugp0vUcMajGz+L4VHOQk3duKBzCpnpW9viiQOWXIPGBPxNLfc7o/Pud1oq3glg7oPJk2S76TNS/39wCa6NvIFKkGWAGjDKAXXp5is/7XgkG8/ApsUnBsWogbtLWZzDpcxsbMdxc7iXWxnZusAfT968evHqudNmY9LiMC0wLzLLXSSovIFXNAgCuK5YH26wDQ9k1SzJw2LHtK+k5h001t0FMDnjHNXevl4UZi+Kv9WLwWAewV9xFgNwyS+R9j0P1FPheXYuo8UQbdnF9FUSAyFDc9M0C+PCmAAs5KbAilgOxYpiwEAWwBiGFqGCAAGKmmOgx8DsM3yBzdTfEBX0v/HW2z0vi70Ve16nPgyw/413HcYoGBX/lS/fJzNUBx8tf16g42FNBOgjS8bePIl8vPtUr+CBiwPa5wYqPQOrrGrrHhio6HWAgpJBiLrIxgswxJNtGCkxAUIBND5gGxy/QumqOnwJogr7Q4YUtP/Yxf4orxDuTx7Sgxm7JPDg0cht9RGAeCi1DXu2W5mpLrtotxuiukblZtfcTRqAwLCxPFCFH9hJChqq3qpj6MJ+Yq6xVdkGOz2AgE9NUvODXQvn9EqhwkuTJIIS84X7jqzxC7g/uwWXGQSjDTaUd5Nk18Ck0xMw2JgyLi31zLqqugzoWPvXMHVZ3kIYaUEvuVa6smI+GecgPQkRBpeBuH8bovifJ0G4TNDJAzrboC2tfEXDe9/5pecOYjlZpxHSArwpwflFFs5gOOAJbCR6yURTMbq74GWSDsokSIF56JMLmEtUWaXS0cYBz5YXhHMQF9UbFEkwFh+UoW0pTgQpjgzl7C2FPHlHwYoHUS9YfcWQG6HAPm7sq6r5EqoTjfSVA5xSC5/HX72tapoHmJ/DaX3wiPyxf70tAgmkyR3YLo7DVgkw5QJUt7Zl8osFpmxOJRs25GW92NXfYawBi8ejv/HDnFGyG8w1Ch/vRqIPnSuVl0lA8BSGL8HUiov7RoDNojluPV5vYSpq7HoOZmdRBtbKcBQGryjGRRwXyPd+FpUBtjdPXtZ4jMyC1YaU/8KfS2+1mWmbIJ/7QMw4y8B/WVUFbH7UQ5dX/TIYrbEMTbE6DNtxaeBtcxymPKN3XkgeTk+xErbcUikXGQkFR0/Zqr/fQ4fByrUXLsVESPYTqAU3t6Xn+VWvNkDTSHZoGrA0bOsVyct0A4Z5JCUJtDKS+OuzCUxpw1tR/uZoULEF0EoW2NSxejMxIHtUe3ITgr4kbYbFMT4zgyZmm8VCgu20nI4c7O2ijRN22FAhdONrBk2xArMfPRA//SSOG26ZMV/ocM/60a5GGHcWiHE+0F+cORUG0CTCwJmLvwyzjkbbzdHjRzRSeVvvR5EUPurXfLO25S2oyNTOwIWkGXba9qK9sC6YY1T0VgWkgVkm4iMDU/7dxB0vPonnT/G11R7awvpojEAciWLijhafckOLKLwQVEMps3pKw1QaEmKIkauhgAqzxGnY8k0nRnNtKTJOw3fPouTmpV+AYIiXFwA6Cg2DE9sCHN1V3sXWUGOB8wC3gd2gY9XLRRKBBJ5Sf8HL8TBQB1IsbzvrQDo0oPqU3RPPwFVY/X4KKA8k2iCLcE5LAbwK8R/j0S2FUzI/DVUUvlf86OG5MvZBN3gLBL0N1vZ67vnRMpla63nD21fs/m6XcsSrDbCzld6W2mxzgEBpCxZRogVtpyDM0yQOoaEOkXIg/M+GrQgWR1nRK9nTTKg34B6AXUWzrO7IhDIf0ISbD/5Cw7LYkddX12q/IAOaxqYr3mFtYT9UYY7vy347GIwEkxXjubTsAF0Dl9AgLVp/AOtJ5H5ECxTlMoz4BVCEiykcwhSFv56FUIAWprR1C8BLUGo1hwPM0JsRd0d7S6oXv1CgN6gpWb0ogEGPStpRCAQD7WC75O58E/ggqIr1JnKB4JMbr1icHAN/ItL2VIrjzy0+A9tiBY7ddbN0Taywv6x6/hQgLLMEEPeGYmaZyczmfHZbaLUS09Lsgn57Sewt0w3YbBi+SQqDWu7WC9C24dp3tlxz/VujqMOqmqZwZs3J7wdOnEQ0C1QlzHVSRbVMfiS1fMd13Ybc6ZQqjQADO0GGhug2G3tVcd+PtVLVg6MgLI62x0ejI0IACIB2gPUL5Auj4C7B1UEiNbps0lh9dr5obnhWMKJI3NoxL7XQ0IET0IXQto+nh7Pe0kj6yY7iR18wOgCsaG/rZyGtQOSFn38B8akI1kHDB//k8ymwkwjX269LhRVC9lFiY1rsGsmBdOqZGKc9a184Z/XZ2kebd0/bf++kdU1Zm87/xmzV5mmIrtUmRyVLKxRjShTomT0OrZZTwred81HTdhz1b5nQF1i9ZThTjzcUi7UvDVxQY6YUvWqoFQn4AbfZAQumu5511fKic/THaQU734O3RRdRUe+F/fFbav1b7Ex5SRC5Q9+yRfPtpw5K22v39r5oBbq7Ebrfs2gRXbfIrbke3b7q3aa9QUUWT53C0GRvl2iV/KtioHOQzmA/eL1eAP3tdurBbb7+Wu7IIcp8/0KHabqV/Im2X5tHq1UCXILiwCe4luEHdpLRiISHZkC81cCllSwW6K1YV5M7Oqp9ODL0vXm68VTVhjOHXHOH3aMS4148qeNL2HOQYBkuZO1wTTdIjBwpx6qJM8RIl6XcDrSltOZZk//DproyHzid4bfUnAaSoOUT0xztcYHdIrEtQJnldK+F1Aw6enPjZ+tNandGY16qZYYIVCGHYu4Mq+ja3lqi8106qqqZ9mLHb7GPDpwfYPpZHIKNDl4d5+5NaEkjjNgNvZZZDFOOM/XLb6dPxA4IIFxKY304FxhUyMJizeHX+eb01SvOUQQYaQa9znQenI9pfj7GbTHJMUrmh3mLKlhTcwxqRolJOlRYKT3tWLErSF5mmHv+1ofhAUXaDWJgZ/9O16sd8O1KZDvN/Js6268BHufxxPJGJeAMwZceHw8F/HLgejQU6p/jtHJFSrAu/rLDdaOIK6MoTHNp2+MfHg3FAwBzcgIXx6ORQ/G/aGrD9VCMf8BfD+HX8cOHzYaAajHhYhkhnB+h+vhHBDRGiD9UgB7BQ8wIGj/qBIMuJ2P9OSVWYrwykNtwLqcWzoPlgL8fg4EJwg1EfyPQSkFPrh7GC5mhXiDJZEP7/NzfFMncz3EpBsENRVCAipvySxRcxfiR0x0YsymXaxquh5iO6FUt5IVM8ynMRjLHxBsvk2BzbVA/TMfHj3tsRQQxB3vmOp9qicW31tVQZZVCp6ZLgErZVB71E5eH1jJfNY3bilDzXTxfZSA6P0jbGfQYPhjCAVMZlCay8iGi4rMi+0IDX2OSbCKSdViEgTY/SzGDuskrEpWaWKY1rFOSvDKYjurSBzVEgonTKqmOMqZ5EkK/zLc+Dd8NQbjPow00SYuwhl1AKcH881ay5DLFDWZQ4yorG8+uOIuLEFfJxFq+T3BtZhlRviYGe5O6AFrApBV+UWRqIOYi3WHGNFTAcTUMahhOtzHd6xpBDWEXhCudf9jnq0OTlZ1ltGQDrsmmOlLv7ldljrZjd3x0qhNz4CovgLVlYDTAs4iyF/DBN46b49Km7VxOTkajq0aKoe6NKozZZSAxNlEkZkmwA78Ok60zGQCBRGB2iFkZlrIUrYAtYuOF0wrCUn6iGYP1aI01X+pFgyQpptZ9oJQCrch3+PLtc6tBe/xU2L8/eftO3Bd0L568cCYqVgpEQomJixDUMOoneJLDQ/a87lZWHQRkrgT323tqlPUYZlPV4FIhDFMnfOY7XsvusMh2Ou0xzjHfc8T4cQ4pWFsgx1owiHwOoyhyy3HatK+mwU0V1eXllZ4hjSqF/U5GIFCzLASbxQi+wv2bl0+f19VDM11G9bHyJfSCcyevYB+8TIVFFfyWM1KDf4MptVDqaOw+AAywvcdaTG79yO5rhWy/2ojvbqckW2qmNV1a3dUVXY9E0cROpqSw17jJBQm8pPSS+i3nTlo8VHPokbf8nna/OkMdlZjvXzrm7KJSGCijsCYNKHudNDoLFlDGoEgwubOWF80cLNe8TWE/Z6uHrLnrKwcknj21M0KXhGlO/Uyl7NdXdHWL7it/LYN3vAsm2z2DR3a+WSzC26nlpvESDRsZyUJOOcBES9sNdqdUen8r7cXKpezMRuoWpZepd/2SBUx+yjI0uqwW2GdLjzy3aZy6fpb5O/tyjLk3+tcVGGXITl4si2mDxYxk6zAGT67RaJWITEv/gy80Hzt0xh4TstGFUmthp5VtCMhoGG9dliI2izn/isqOwQjeG1DUBOjlaFZNiXlt/ZAUh77+WTzk+NQPLngCMLuYZJZfGupWr6QzMbrv+K+dQBkXzGKwMW2YLmriBF0LnLscVWO43CQYUMR5TC7HV46ZHdcZaQD+gUtclMc8o+aqZSua0afZjK7TJq2pQPypG+3OgfXMT8y4RrfDVQ9fuphhjLNThBTHYCgycxEt6k7z5Ut49IaeDHqCISorzEQFkoner9JfgxvSFbK9jZR4qPpjq9Qw0/XgYkNhdmaKvzqkPk4eRmbNCXP+dlbPHFeHeRcTbyJC1aEy8Zq+gI5CNGIObUNGicic9pl1vKnLXvzZogPNFdxtmGF6JjaVZDu7z/nkhWUq4+FKMSbRB+AZqYQWSsfoiw0wA476XC/c7LehCMrH7doFEzqopcncx6eN7JkRZ8/81392Zc9wBg06dB+XCoLiiG+3mb/2lrNvr7C69Vk5u18jpU9vdDOz+lJTy7II92ZLTnapyQZ0C/YIhbqVgOUb+8/A2XBa6q4MpFAaGO6Fa7oh9Z1yg466r9Mc98osPNoXAIZDo+jKB2qI0pWvNk7RPqvpVO2tohhhqQ+x4iW4ZkNxcuWuwZBzxE+gDkbmwEokcdWkqFroGl+1eG83t22ZNvj+jV08+ZmcA0MB/wItJrP3kvYHoyOWIU/zDtqNH2S+crPX4IfKSlwjAjpHWpbYgfK7zblYnMQfZJbYPuiw8agW1MWUt9u8MZu3YEPcjodiB393Y4BxmzMGh3QF2hWuduWznXrWgTFCQpakto0wdwQX/M8xwYW/pnHQsQeR2FsppWoj33dg4IwfG5HWv7qDeGhk9gbw8r9A4oORwBRt27k4EjcghEAwYFW4W/Fd2+eFqqAKUBPb49HxA7C34HdrPyApYUL48QMWW/lfhnWcRmHh5SspC63P8Rr8vAz1OW98HeJu1yG640O137XmNp+GW/CHaVPnKnlPvgMv1fIG8FhiPCVH25HyBzEdaZHEQSu2Qi3vFQjGhoJsOfPrrNDHuY3dc9RGaz+rGjVKFEaSIUPoHVhafXxWYwLsV8kFhPay3Jy2NviuH+/MoGYulzkGs3JAIvbAzwpKIlbrxkNDzeCiRjgUlOIrwepEExS33UVOKxKwLUUJQ+5Y2mi0yEloYWNFR6GKwe0H1drZYY7Q9VPAZQAEjs1BO06N/7sgd1VDOYHjNWr/QUglNw9M1hpaEY85rxZTRAQfIaFBOWAstJlhhv4ARh65owdXTXmExUlYT0hUArGI1QbkZT7BzDbc1A32MhjreSspGWsTA7nEX7Y1bMZGdO/IPQmhB3+AKRA7QxIx8ABlUvmU87FDysfGnYF27FSdRc+NQCU6+EalEyzd7ENVK9kU9Q06epsgCB0Wuw4C+ADwqIEh9bjpHm2KS3x7VeauofJHDjCErZLh9BSFFbkVrZwIxeAA8auYJRTEqFwXdFxDHXNDo3+I+7E9zOvPD00ftk+lcvrkM6inMuVwIHgTbOZwc2DGyDP0vGSmQFRIwJ4hE9Ve246Nz1ulOrvTKtxrW++wHFiztZdrJKTylYu+x1tZNDTqGpgyCDxyKetlqW0XZSQ44zlj2mUkt0DguzBeesr0CTZApGiBe7q6vacKzqYH1oQHDkeIWQ6ehDnbU2UegSbwVgkwKypwukBX6Xg0Gu1px0syXNxgQvHmyYqcyKjpU6DvscYsUjqDgzHjdPotjfCQMZ1NZ31NeJS3MER4RZbOmnBpPFAhpWmDlz5zGaecw56Qm+pZR8CtM89lz1D1cAHDYEX2zTRDZmzwVLuL8NbjZJy8Qd+9FaJIT3erfCd+jvcnLlDoi4+A4YUudm6Pa0qslCmkLFEJGBwgfq7eT7oowBAkio0rKQW6flpJrKbUJEQ3dkwoYYe76fIy1slrl82Yp9pHgad/gMNbd9E4LIyujYqgsnfzZasYnfSiRm8u4KtYLfX9Up29ctUXsG0BvKfj6mVqewLGoGBWpim8LzHI5k9oKydG3n3MayjX1gY9O3Z4Stomzmfn8un+LTDscGdoW1v7FYJQT9mfMbNOMxpOAD6bQFoefFcwZr2JClxv/Hg9AaMRrYprZbBSS25YyDVwPJuoBqVUlohONmUziuAh84wrOxookHMM9+TywIOrckeNSjDcnwl05Yg/m1a07otKUGx0gbbOGXRrgtzTeK3l4qqZyhLL28JOqwQmbquZiFTRB24qInRxdsudGU+GyMopi7mrXrGv2oJqNjQKEhDqQfDFKeSO96CjHjts3usn4qcp+ESMk4dfltaVFCuZNTpcx2mj8+2xdCZ6KbiNZK/aU0RVd6IXFetI9ir3QVXrgn1ZHXKdFjveE2o3ozjMVtBJNduEv7qA/J9ZIPkbOTZK0tfSbPasnigK6rWp24k4WIufOsO9uTeDL1qFcerLPv0d68zl6VigYYXCM/pV3B4Vd68l3+gTDbTjowQ+mpb7lm4sSy224Hl0iGGyfULM0IgxxBOJVZKB5gwSl0NL0/LUQ97S04ruHLQQ1KlkP3tRCJqzmwN0xL9Mm3YLLolos6XDQG4vebWAmmxKXnidJ68wo50fXV+xNuRzeHrOkSO1SHoSd/lfO3t2kt8Tv/O2iCo9Khf2IqMTC7/Hwx/9CGbBdR3B6ZZgbuOGcBElwg/AhpVNX7704n2Kc9JElHya66gIXpM+ZhmEt0DaX+x9aKLB4HJtnwcd8BSwrT3kDQ48uA5H5M7O13pYxRp4H83XYb1E7b1FvruWu7p5laSFGWhAMaiOX0StNDbEfUkjdPhgHW+o5E3qaMf89iaMr/TBL9cukAtJc6e/0BY01mxXoHt2OZmIH38wMLiqHSETbNZpbuMQh3QkmwfDV3ubOxpS2F+5K3kbhOC1F9jA+PiqMlKzTazQx+FnygTT5yABw6hL4hl1TTFavm7He729wezmGtWwbdeoRli3qBtyxdQ1ELHuB6apjY9PHtShaBUypYNJoN2ptdrgamkY4EkhK8pImo7dx5iJBSJKTkdQsD75UDSZWr/4YGH5mTXkB6j+dNOcCKC7BEgucLGyY0xKpKltx5mMQrnFFh+bhj6FF5YZxhcifz0LfIEqZCIUKmIPY6ChH3UgnXqmwhJUFPh5qfEWgAiOdLfYv5zqPT8W6yVWICqsMWkei8SH9FQEZZ4X12lQUqp9pwSq7WC4KEXIoGnUluYIqp2LS4uua8cNEWWokuqGi/KNWZYjBFyUr7kkXddSG2i9HovhFRdq7OooaUAVrO65eHlvVlKEosel7tTI+K7WjVEj193MACUCsYFYj0GJ/b9NWFCSKi3vWGY3QWjxiVikFydGuFixeMmDjj5l1SxE7GqWaRtfrErNSsTodcDM8WYhemKW+tS5JNUtVU33ll0MczlNLSlVEsjR3m+n+2FK9m4TqYZONChAHQyaEErbo8fMah1g+oxSWvV5xJgBSGrWmD4U7XgeZThXbDCthQumOmrAIpF+s0yE/83M3W6zuZSPVcgBhST8L4Uj/ynlI/8Z9uAHRSX9NsVkeVXJSvW3G0wlQmdJEtnlrVPm26gXfONUopT9BnXn9EIHSashzxJnqOUmPeNrp5KX9FTdOFpw0u8e+JUg1b1U94YyvtYyDy0W01Yxyeq6JG8+uoe84trBqdVLoMpLCx6BAOmz+d6UR3XKmA8DQovuujOXswa4EjzNM2idxrrW3lO8SJTlRbYgcWZ98+/frL8JvG9+/eblN289C/egXnefI0cGY+PwuMaprHR6b611PJEaTz/Li00QJu6qWEeWQ/s9jMqoHrHbSlOiLLaV0uU1uWDSCPrgyVzNgV7wgV0eoX/YNwos0jsMWgXsitdNPtPW3HLWaT2DmVpeWGTfex/jT5TE6hgK5UyTozROVLwn3mDwnWRTBL7mSHzDh5WgJ4MbOfAYeUxEQxPFz6U+JR/P3c7CJDMO/UDaOLtUx8ahb2afXZbHxnHEjnlZ5XchMZyVp6hRjQcVrjx9CgKuHsOrkfvjCE+wrxq4j20aFVij64pljTt7wXBqJ2IV4fzaBsdohgJ2jottOWVEJWljrsANf7L14w8k4meYpCDyDczNmhzFAFN9SOrT9wEYfdmy7r0T8sY9O96q07PI4IIOYK5cjsuGbbK4VRFCGPjDIfjo9Q1JYxwpJc3SiMYd7klpcvg4ezOo5eM6+S2Nv2MDA3bnxg8LaPGhU8dgCh6YbKNwETclF8ConQ5/tgWkNTBAedRVET6Ryy4wW6qY4lwNBVzn054pAw73wQKJO45vBu6h3Af7gLU3JTUXcaNwZ351iZ+8tcSr2iWuNbBWHrZ6WZ7meIVB8LGpLfSB+iDs59eYUNnIPDRsxhMQjGcx5sX4E/2RC+BZdHBwzSZRJwtZjcCkbqFDwTQ7d2R2rlFKH0/ZGEIvcuOtOjitJvpIZBrycouWVd7lWZMN1cdBOj5K4cBAxfOJQAFPwCpKZgwN6XJotoHFUWC0y8tDn05OLeQfq3kSikWO07YK71g9m+zbPwTuv/6TDxSxGqsfAObQ7rLem5jHdOxb6/qSk1UaRmdj/bNj5kpJ3xBYeb98KifMOg/XaUjox8zKjGfCX27eyyzJra71S5CDfLZbeYCsjHnjmRGKtTHihGnXyiZUG+ZfPOmCeEF7HylrO6MlZrQvcm6FdmaPQSxHUUL7Lec+fjWGPs4y61gNvSf+4+FIUHIfr4pnuPcED/DP0w1l+qnn1Rl1O7gOl0CEbkfq2CLxaGzT5op5l5NUFf9ZnIzw5wuO4Nib52Msw/cCOHDFnju49+yNQ06ZaOciQJNEA3Rcrl/LSHD2LrDrTCjua92kvBN1NDXyFrFUN+zIMLZoqmlu3GU0s5yvdtAf0xBN+v6R3hPHjkHimiCREHNF1Ske642BciJQHXzGbyWBJQmsljh3sjTZbSMSyhbNPu4eJhPqY0mdk+EnaPYjJ3vANTVudVJ0O02kI0Xkq1DugVTbw3y9s89mveVnc5AgvXPf6yB1VW/4RzU/yWZCs7iMOfkgB1NfjCc1uYPLpPLzdXFlhE/6vGt8GUAvrpVZz+q6RiTjIRv9pLZLC/8zdZZ1EcaFVt/mcsmLJ597RlVNt+rFweay4N9RqfqHVgQZhyrowHYASZwOwdB5cEwPnvvScFhTc5k79bSGu4+i2fpUC4DhB1zq/ELi7oO0n841wdQpXdP58YSIgHZfHi7DezdQHZyPcNgGo9pGo8NckS7LvGa/G441+twP67729+qRMtYGbU7mwBWRUb35mvttgK4eVsDh2cAITeivyi02MuNlcfXBOTSwyKEBGcRZA/VvytHBRq4B6ZSOWcaEziQvP/U3AbccjDyiUb88yXyIZh85SLwKjR/14nhhBbCY9R0OGqfNwNEdVGp08qI6Srb8Ko46/pVSA3jmJpQrzz0ScajX7/GcoN3czJcnQqdlc4vHiEvhVI/jIAoEJ1DpIle93/IwA6eNQtpIVoV0HLVaocgxnMxLFPiJAvp+APjncG1hpwIQXHjNnSuPWKD+VysGsgXlUz3mINGYsFW0ok6EXBfomWgQoyDGk6G4dQY9RglKPHC+Ru6PP4Cso3sMidRqU3Ck1pNtf0+2rZ5s9/REv0NkUfl8aTLIOcfogAso5EjHroCcBKuVvos3QQpSwVH+Lp7wZ1mYVYRlwJKx/uaKjhJEvmYOtIfdrxOOLQ+/t2tWyR4OAfwXG/DU2kcyNKOdHGEt1/eboaZs3s2bi7s+6QU1O8JtJhqgRHOc5cFZ5mci3HwzAyvbbodNVCnco5IOOX3CWKDRKzNqGcNYeCmXWxqr0Ho9pVw3wYnRYoA4pV5BWy+20o+lza40pLkqzbK+vEUXxdwztGh9ughPOFTMXTJ2M+EGv/apRNKr17RM5tO3I+gDq0u0q0C8y9g4TRtPsonR8W/YKveEv8Qq9O2yf764ILoOSrqO637yYnvwRP13TVbvhDHOhixPmi4lfxOT8MCaDXWAVpmVjnv99N+G4u2786FIQ/kBj3jl73TsxPnr02rObv83kmtBBhqnNtQHP+MPbi2kyx+n23dKHJX9fiqsP+M/YwzNLW7btT4rs0pjmXrY8JcLbxa1jeSZaWhVSx6GPWSug2h7aBb1j6rMDrLPhqLjG1OAwBvL+fw96zpuVN/BFrbsGo5LwKRjfmnS/D5WW4iGzbUT1UjjQ1lQzICS3u5v1qMZOKTx22brt11t3zb8Dxqa+eEgbSXyF4h0plZ4yx8Oqr912ukMHmu+KAk8jlrg4UnElnj118YPcvWiO42BAYS3Pf4H9Pby2kOyCm/x4o6x0PceoeHLKwcQVeBW0BZc36Oe+6phNU6jJnYL3iqjjRpplXF6zp5BhNeITiOUcEDBnimeHRHltsNQyyAQGhaUt00mwFQF9nXygO+F/pS9kWH58QWPvJkpHXpmFzqEPO4Jnpe1FMSyHsGEWlUJxbpliVl0CFg8okJXqQuLEcNvJzF0QlSftp5yjiB/vXx6cWnxFR2GmG7wAWX8V/iuBAgiVcmQMGW5gYIdly/xUEZOCvP8fB6GOu9MEdEU93QbqSy4NbfNrQo/KDsRMsJ3b7IQ5hlFs9OV8vTjY1y+Wqc+WCEcnwFFbrD4B5JHm6L8imrDJcG9tdT4h7QDPh2gZJ3j4Vy1b+SUiThBmE25ephOP6RDIs4p4+la7qbXWqXxJ8Gni61+QAb7lO32hlcA3nmYrzw2cmyo6Rywg2urFN2maOusSkxu7xCT1GZTTBqMd4cc3idYt06PMNp2Ad06Tp/oOkTObu+Ws70QD5F23VJve4DUu0voGd26tEohwLRAhoBuRb/rWRs0hee2XaTG1F+Dpw3muCTpy52NF+2PaZSfBztABID87jpRrVSWKyUgSpttW5MUBzD7F20e5PnoWAHpjaxpD51iGFfNM6cbvqL6rjS7nV2fljZNmMWeLKDFlraJY6NgL0QRpnDaNfliGH6t+O+hvanhRyeK0elaed8J2I1kCSxJz/4V0RTO17JYJUG1m9acOePMqjVKdxDxKNzhxqqnKEAlOsHvn2FKx/ZBCSLwYfXmxYV3evbs/Mm7s1P+UiiK/Qi/mzwdE+F9aOeaUlRCffUcKBiPbsT0O3Ju2/OOQLGTRotv371+c3ZKU+fil4+ws7Ztuev0AVp36lRBS8c72IfCb6X/2b2IiYG5rhG1Sn9QzNLmuwUeG11hgLdN4V0tfS2T4EnPFSKRh9SejV/Dgo4Wj/BwMdO7PPpZfa4bI2MuYJQWtgNfe6JU/+WL8zNAyauzt+hKVHufVBRiIi6r1SZKw9Bu1UQdhqiXZbhQt/VTD1S4eKJ3BYYsI4JCcTBdrg8SdP52V37IksA9NsEBo803EcHjEr2AyMHO9KE8COmkPj4qwOfM90JZ+ymnDyiPGMHg0eIVGP2iFzUcvQt0ro/n44nzAWHqIXbojAvgiJ6fPxX69cHwbuRMUbOLp5FbCIRyERL+PBm8PxwYGCvuYnZLwBqde/b0HwfAKR0p9wckgwu6pRORVhtA+CEQ9MT/gBP/EleCSnI4eBzg3CGIx48AxO+8LUzmKkUjF/b561PnYGBKRqgjP60nbPcYrGg5xoqnihd1cNbv+KLGVzKfy7iHqzop+NiAU/IS03AfFLU0QWSFIWRiBBzHG72ygj2hTtPpz2W5/fCWIdDXyaMRTVQLXPm2Bwj3WGP0d444alZUyPxUbZdShos+de9Lo11GhKvDtNJ7hB4NOTg6xU3aFodc9T6hcmsQ7bhp7iidr/FDBZcqHfvp+dmr0zP8qPIRqZqjo4U/L5Jsd0TZjpuUHtKC0IguL9qL3V2RfR0bRFZNd2TIHR1xAyHOLqEHbgBDHDZuQYWXUNlS+IM7RqFFp4PZfMNgGafqBd84XeDKGCnjXj+hfTlcubxnwHoXDr9Ud52gyznDIY512h0/Uql3jL6jSkRTb/v2JkBJtThWQWzumq0KkBFWtdMBjIjFUkRDHam+8a3Ac877v0xLvVtu7i3X9nQLVbiIjiqiQpUSJ+9ENdR4Va0eVmYFHUde/15zmvHngtTJOu4FOQtAuTiAAKhlar58cXFGz8EiNZ+/fXf6+rd3d6VRYMxGbYKbbRZ0GtHY+BY4ppZOG0e/YYfRAkwzl3vTysq3ntBxaWKR+WtJKxlUR9XlRGI2+qxnGBbusBrXZD2DX4oerJ3VIAr7z+B7B2YOoXX63uty4UUJBf7kGG9wmJbiv9tvrGrZmCROOeLuCZjYtKXXxW8BpPbYwSzxB/Tdkg4p3eGT0snGccPl4bPo1elyhJxMHZ/WXmpTAo0/f9dvYpjo7rcddKkO1OuG7HpaU7VAuvPvWCNtQur5xFSLGC5R4jnNuWv3UB1AaF/L3VAn3dcPI1znfcEWcF7Ez1OmbGwIYGhc9Oc9Kj6gA/H6D1wwCKe5EwB4hfL/DbZPlsUt7gQBx91i3UHEUAvOp/gFdZSkuH0ZZdOoZ6GFoV0eYa7npL3JnF8PBgPv7NXzF6/O9IQMBqi5QV55Ml7ixzig4DTNgNAV+pZRMvMjoaoNVLc0lNbWvgr8WQWvjEeozmhg/x9ljtx3", "blender_rig.py": "...", "app.py": "eNrVfdtuI0mW2Lu+IpyFWSbVZIqkpKoSq1gLdd0XXd1yVfWMYbaQE8wMkllKZmblhZKKEOAXL2DAxgB+WWBhwzuAYWOxaGAX+2DD+9b1J/0D9if4nBMReWOSoqp7bGwPRpWXyBMRJ849ThwahvE6SEU85Z/YpZgw82XMXS9sM1f4TCRpBjdDuAnSOKRnQZiKSRhesJA54SJkEZ95Ad+LsonvORyeBfj/NOaJCDjjDK6WIoFPWRawNAsABFw/9cPMnfo8FpZhGHveIgrjlM38cKKvPyRhoK/DRF/FPHDDRX4n9FUinFikebMkm0Rx6Igkf5LOYwETC2b5A28h9vKOadKMJ3C1tzeNwwWzmHoHs5l6M/VQBDBboV/NRGrLJ3t7p9++fvOOjdjYeMoXgJHY6DDjaRjHgq7ecT9zub5M5dXX3PPl1VsRhUmIV294ECZ48YpP8OX53tnb5++ev0fYqz0G/92DNVhMYjFkZsSTMOmw0EljITrM4YD2DkvFVZrFvG2xfo9RE1ishQC4AGMRwiqG7H0WT0ICZ9ClAdCOO+zw4VGH3e/Bfx026B09bHdkm2ci5b7PXWr3sMOO+4MO6x+vN3zLI0+2GgC4wfH9Djtab3UKGMA2Rz3V5cP1Nm/4lbegVse9WodHvZP70Oxm7/m/On1z9s3zd/az129hbsYBkh7Q6sGrLLjOeHD4rDs44EkCpHEgrvgi8oUNQGciMfb29lwxZfbU80Viuu0h9RrGrohpFSWebB5Aeze0Zv4EF0U9/ZhxN6k9A+6pPYm9mTWdXBVPrIkvArfUQq6U9ynvgQaR/6ebAR/y1HNCK5x8KH2dP16kfukxEGcsEi8MDl0rwTc1oJEnPvF89JEA+tA3PHY8WGZ9CxOwvWAaWsiMQIn4+RSJaAyfRDydWx9CLzDdDpu22TSM2ZR5gcKhN2W6kbjykjQx179p5yC/ApjTAkQC3CVcE+WBhX/WvjX2jXa7XZ4Y9DdF4YSfA0CQE3Q3BZZ1k0svnZumYfnhzOjobkFgZDE21qSQZIsFj69NnLKiBw6TJQwAq5sG/5AlqUDuXN3Izn3gfUJIPhRzCpQLOEQpB39fnw7Z/v4KYYxbxJ82sn/rfNi5kfy6v89+/sv/CKNxM8dzUWQaOJmi1+Iro82EnwhmwMVXeY9TQ3VAzWUnrQ5r/XmrXe7kp/+hBQNbcdkU7lvtGxZd4UujBFAkHzPhizRkJcBADdjaLD2aZ0g90FkPXsibDiOKYiAgKyBLX0Xqo1YX4LWx79LLRMwykPGJnYYgctQ8blhSIuOp8SKMF3zIVq1F5qfeEsiLtypIK79QSGv1gb7SsHVTGZbZ1PUUwVOvNLr3Cm24lpUptRKv1q3CsO0V3QbYZWM3qnHRURn4W6BbEQ8rC0CPNOIqCPn5D//A3nsC1FIyZDR8tnliBU4rk2EmaYJ2fUr4MJ8NzKVMSFtmlfdSn9nXPs2j8eOJn8+x+eOV2Vp6rgACa7GvWJLGZgMU2aLVBi6BZgkzBYgU+ZJFPihYwjbMqTLRtc/zKRsVXiPC3IVk9/fVGkl5U2Fq0isOCGkQJ0rW5PLE4lEEaDCN06IRUB5MBoWylIEkUCpQzpU8dCoiy/F54k3BMsNWRlsPxCm6TL2IDIOVMc8WgBuYPChd0AsxCH2OPTqg7eIsEqTXZZ9kxB2BcZHypK5gSMk4fAJaBtvLK3bAJiA8ycQBLQaiBd+pq5scAIiqGY1mfEHa4ALFuYkqKw44mUWTmH+SBpIDIsongHIYtJwOTftCifg1lE7RkhFOCgpXmq8gOgkB9JmjKB4etNodVrlt38Byrk8U//sKhT4zl2gpr3D55RLJuYCEI3muZqbld3udJMiosElcbyQJrVyAoGlVYB4JtFhEHi/pmRKkqqKR1rkPiL0GheN7YinYhkkZ8zAOBMGXmiwE+Q92rc/M39etoN+T0E/553/gbP8pjitM9ttGwxz90L1lhsY33z2DJzMRh2tEXxkpyIILFKzLfIKAaKKZDlsi2UgWKXo8t7xULBKz3YT8BQdD2VY42Ty4t7LBkGF7REyAEhUsSOagf4EuUiLRpcRgExKWIgazTaCo9PmWpX7x+ccEfSoAiC0TpQzGrQoAucLLz38rn1Hrz/8rgA8PHBjkAfD0DBybTcvsexMRF8sMilvyGlCIZDVgcpD0YA/AXEGKEiFnImiamLQAbpvWK2qlZ5VPqvRx6xwWFnFYmgd2PeEgvB0hvUwACtYqbyQzsjBwNUGd4Ny2jOYst1YcjowRoQ3M/XxYdVAS3VEWpDgHb/r5Rwc9iI34NVOPnLIZj5FEWRqCjQnDBz8YQSMakixhS+F46PhtnIxHjg2Ku36bPWH9nScENq9cX0GdKjjV6amHhHdwnK+u2eLzH4GmRYyfllBc0G7iBTZ+DKhBAkEka6bbPrQheKU99hslVRYkzhIPSIpxjDhow3MDPrXFVmKC9ZHgPPRrlj/V00AZcanHqrQxWIqkVMbnmynl57/+m//9P//AVpc3CpByIIwfgh8CJaHokzY4FG9fvySP3TjNpKrjmVR+r0jJ4pNC3WKkIPY4yYthRd8WqtV4WtWjuWbFr7+ra1QMM8AMc3Tiq8ALwE5BbSv9naUnLkVsz9OFb7qjb8GI6LC58GbzdHQfvWx8rx5ferEYveCguzpMpMCN6vkimdGVQpphGM9lyIgdPmMmxlwE+I5tJL0M8D2N+QLIXcRgYZVEJYjnCKBSeAk8V7Dc0R/GuJCiOnTkXFBAdHGLV2mooJWF8wI9WyynWi5YyceutwTb8doXo5ac8XAl/72Jrh65XgJG4vVw6ourRxxUUdAlxTF0BEbKHq1TpoF+oTe97qrwg26JELou4M5JwfoagubNFsGjGY+G/R50NOHOxSwOwXYc3usP+of9h02wJ+RPdzFClSXD/hF8SCp8eO+En7icP5pCp90pX3j+9TC5Bgd10c281pMGUOV501eJ90kMj+5HV60n/+e//Pf/8PgAGjzBVk9Q8ZqwvIh14102wfAdJwcKFEOU+QlnTR1MNHg1RPFQnAjRevJSgLTmMZIFyu7XpwD+uSbOxweTJ+jRwvdyAPRXcyvG3bLYBz0FCx0nefDtYxamgpp8BEabGukIBENqYlzPwj9mm+2jqAHXNCckJOmCID5i2GFq/NlytMIXRTMk91oz48+IB/p5G2KDtUaSORB5NDyT7uVEktihcR7IYKPNI+8Ag0+jVZ2EW2USBk/iz1cf1dhKJKyZKXZGrRX8vWlp1F96bjoHCuv95tE6cUtqGvZqlNZEZevk2QIR7YeXQDyZ7ydg8YjgEbuKuwkMH3RmN43hE5AyrRpw+VnxEaywHH6NSKuciRZUl/hvGOPwH4HiBAu0m4bR8AhJ9jFnIGGm+fxTeC/SUQu8SB5ctGoEalQp8+Hk4WQ6fVSwQf8QYZ5OYi9GeRXxgEKe6CZEPoqnn//yrx4f8II69/buse4v+I/NFFOgriZCSQDiGy8Bt4QcdeA1NLtgDKCDWAhenCcdrzy6Cb5+Fkj/D3QojhPVvMNdtFUInCkVbPdJ7rPDZa4W2hZ7Cg2Jt12xhIdLNEcKKQzMDl+rGP89sCXYAsw0WGdU1iDiKNIN8LicgMVQ68DrIFl4GIJETC69ZcgCD5oj41KwE+Bbe3/x3dekIrU+si+5l5pJ2mEBkAa4FDFYLiJJysoFvFKMkCFaQFlARxP+IWRgm5H9gGOlcbDfI4jfs0WIY0W3H1rzJRqPuV65nAPzUVc6bJikYwPvwVXI44f4zAX8GucFr+uBmTB7s2ednHSo2RRc8A5qNWcEvgk8aS3QEMHwVlkodbFxK+2BIWn1phiFaZd8cWiX+EJEAPc4t7oQOphi5THE3ANROIut53EcxugsII+B1YrAoS30TCQqA5tRaCIZF5hM5uGlredhhxhxhbVNPNzy4Wg70eIBqcntIFx47i5ADRZqWUphL4A1cVIZPL/AGHwdtDILqZkFhl6Ado4wYehfZ2kaBpbje85FG4U74B5oKimLutU6vCEb42QAtRhjvpAudUFFuFrSqFHTvce+V/NDK490EJEkkctSfBoicTjCz3weKx8H6MtFqzRgC7EAmuXlIZXtJjSAjOdEcoobXGCC9yFSXJJ+/iPS/09/d4hEoC0jtaPw0z8NAasJm4BKDTb7D0p2ncrAPLDjWeh//nEW4maOYsUgt6OLFVfGEUXZJRYiWJv1CHx5XlFDzD5qEyfopyBeUVbi4yfsGMx4wj1agKpjsD4im8ZiguIOgJdRi3SYL6ZwTWIcmGUuhP7HJr0DrwDPS2FPZh2CUWxq1dEy5WDU0x4XtBQCZoEMbsvLWea5yOTUlbZm4SZMeSrQonVFuAaRnoLzAFAT2iJC8ICQlE98oXfTEKT0/YEE82uwgPmFUAHDNcAubZqJ0fs4w5aw0PHIyGU3M+PPf8TNsrZBsm4RpSPDoP7sGHfRuLK462A1L4yAhc60KGqXpCSRYx/IiwfgMJTk5HUh5HPXS0aS0COkJRdluzvHbCF38A4oSe7DWvgPGl24dbe/f9gH4daXNCWCGTQrNkjNXJgRXTAvIbIh6iJC0E+KrpThb0rckCFqtC2geC8yS2Z9gzSsGazIi07sqUcu3USO9/kfA6ske3Op3rN6fSXHwS0TpD4AUbRvGCivBZEKdBH+/G/+W1l8x9fVccm5jhAbFra301DuP6pJSRIuAIgrR8BUn9M/KIE56POtU50a35LEijKMaMJo441jBeUgbozmdSj6KA8Z/CAvtWl5zHWmbUvfmT5QYD94SBxqM95KwwsR2HNxZd5XJnBKLvEURDggGXcs0S89kx564JLbCksAD6V7SUovvyE1PGRrTKF0NqiFc2zllT5Je3BdUr4y0IyGxxiGeo5jTfOYQOi7qKp84AITm7THw+6gV9K5+NCKwsiElh3qAuStZPQpkQ/IPODzEm0qswC33fmVqe5A7vohT6FxaQcT3y1kywJmGNigjoFapEHk1iDjVPELt/JUWTA6boG3pWFehvFFmXvWSBZBxGQCSRqIs+CLRfk2laY6K4t6LdvJjRNXbS3gc6l+KzyMuhBVkviHP+1NSqAk9W8f5RatcOvHSlOM5JqrO9qqKPSJVKPIEreCo924EWKIcNdmj0fg5BbG8ggvbgcjNdZoEoZgwxQqDNClaW6kL24HpvUcQVM37ds/UwoR5XmYzAxp6MYmPUY5z+NUbtwb77FJvvk9lxrU2AFZhSqlsRX31dyBCjflDKAe3UU469wi8IMFcstaJ2TFUxhglV5HwhRty7aRRW37RsrntU8IGH2Ug7WkewhWnmNWJxLFkjTyz4rXU9y88Ru4Xbo3AB/tFCkp8mwp6z1dmcq1RukBcogDhwdk1qh1UqMoPDhD2qMlJ05KIIXfArF5REOKGJDKnWpgUsUhjblHFFKybGUflBhBCSuaPhyfXxsqZkmm13Y6USEbZTUdkK2eb7IhcPSGQz8jg8Go0TXo4SxyQa6YCTjUTirckZp6rSFuHDd0gcqG7+8zc7uLCH76GSysNkWUK4/GR7UXWny0KNple1w1Nwm5u1mPg2FhKoL9oJLGMHSngOVmIul00owod6GLkte6bl3UDbX3yjIFt0sEMm/QDR8x1TPXIRJtp5UITO+v7U5ijWQFYIxOlQYGcoHe6w2821c8bW5arHkV5B1WvRbCOgsDT1A4I98nuWalJABpjpaWnpj+Lgt/OCxgd8qgsaPYmYPjnPx/X3sb52Xba2sfl8W3pIYO7fKgx6JIooNX+Ey1WJLJGsDKmOa2rDaVkgZNcPUAlOnWlIi2qyuqS5oZhnSjjZkXh4f3e0aeqEaPK956W1uWCO4Teu0uRsatT14kkU4Ji/DYHH+Kzhtc9k+REoTjc4yp5wmOm3kBumyiahWFrlN1OcDSqd7t1UyMTkVcS1QoFFiL6Aggy9nU2ebn//Rv2f7+Gyl0EN1IIbdE0fb3aRsOZ1zJ46vCBr3lgXmhBqxZZcJTZ27KocjIgTISt/KM8k/ps81UTq7oksceWAvojepw3wYHGZYZxuCH+BL9mfF57p94HUmJIsgWwBiwVtR1uyEw6bEDsNMD1SAPSr4IaeMXcNS/OVgVDW7KbjCQEhootAedeAEQM5je5hTjMrGirKmFJstmHyIu+Q4Ij5A6ajLOCc8j+ltlJsSDdmGAYZH6z6stAEelvVmgmHwfZUK550AmeNe+ofTK9ZfxuAVQW+eV2e9g41X7/c//fmO/ZX9b8Z1c3cqGcTjTVLi4tknPJmYhlF9yXwDpYCRW6uAEvCEeu7Shb1J+I6y6WEKTUAY7oOU0xu3HdhHJwd1Loqecltzbs2xl3rv13ffvMbtapdyid4fRYWlZtdeCNRqCl0gJWUUdCnkvyArK8RaznYQvylGYqi2nakXBzCCBgs/IcqRH7UZDb6dE5IrU1RkYi1nNukc86rU34XWHrS292x4P+8fnViwiHwxw00AtZbA8W1kRAoEaD+/3ztXihwBULr8plukQBcg7ksTPeMoVGgGltIzur7mEextwpVeQNBQPrs3bkZiv162OWZ4svdu6qmw+VNSIgzFgyPICV1yd7zXQ3p0TA/ShkXmWen4RS6Fbywmj6yosfQNDoSW3ScXadnutDwzZyBFKG2Ql408VealeYYa95YfgaiElrA++moqvqakurkqqgCfJP0/75k9rouxmllSg3dXQ6tQtkA1WxxwNG2ikKPET8vZd+ZnQ1szTWuWAmDnOxQy6slnsiFzHVGlRMkKUVBhBBzVCMP/zh8sFjFY2tJZenGbct2mv7LoUkUDhTp9ZTuZyC97baPAgVkwKgZVeeonNl3geauKjfUeIxX2FXkk5NxlM1bSIlUKRqyL+prCi9g3q27enb0AhLxdWlggXzCPcr+gNrT7YjgcNm28ACtrKVO+8MRqaL7/GnNOFFYkYs3xufkO55C/PvmdZwl0+bIZlzsb9czBaZ+MeEEut9xW9rPdSSXwvQOVb85TQ53BnLmBeaMsJi+5ye6bRlrlay4fCmJu4QqK0X71+954i9KmOps+iTF8uY77Q16VLp2iBG/x0ne/FysipprV18kICWSKRoByILuFPz+qt0yUKyOJIH5mVRgB86/FusvBYtwsOa3zdhcGOELT3ieN8LbjvSJqkRdfXtKid7futOBW0sbNYEJgoBFlkuTG/hN5k9G3kJMtOEM4Fx6h3EGbA48n2oGQyF76vtgMdHiF0O8zSKEvVQwwk6Evwc+Dd6BBDbC5c5YDxTNJYhpOvpAC/omNXemtMbtuYYGUabSA3fdsxSknx6OJAH1FHYp1Duw7lZe7DNREjbljw8QAfg3jh48Nz/HtUgEB6sNOrVEYz1Vf93uCIyPpghR/nTyRNG7corRJIAzffgavkJ7dKmyX3E7IOnNREMsD5AbDREreH4Z+cXwHx8FqBgktbPUfVgIs+SqMiS/TC7kjcInidPV4Ml/hlfGHney1XhUjCXMb8/XjYPelVglEIsJPPF64WuBLaEaioi8IXiDke5MhTasyTHsWI3oYfQtrDdUPG/RkeyU1kMqXvYcoGsM+cX2PefxG1q7BX82AAnTn/7u3gG+njdJUcrkqmYuvJGzkzDPlgliXcgbQnP0llVW1zjT+Ek0Q+5ynKrAAW2cCHeB7P4E6KsSncWkQ7LLBxvninTup98/pNmUBOHigKORlIEjnpSdI4OVaE8PBQbVqCAybP+JkGEqSSivDPvePJ8WTaw8vfoLFnGr8FLWNoWYktTiYPJtOpbrFXt35Vc926Pz1x709K8J7K/hzVn+jx3mFvI7T3KH+1HMb2R9PJoXuClz/9/VPN/r/rsFcwG0rxPZHCFi0MoABnDu5pNe6A7mdadR3lXiSyBmC4wzIyHglHBS1gBCGn/kKML+ls7ZyMwvm4CxLjyQgXptIKsyRhqsdHD49cCvnTZ2QPQIdlBw1fVD00Gm5157PUHpXkvJY8T9NPcdaGPmUC0ux3IAcxhvLwhIRZZ/UKdPcreIgZX1ckKdsk26QO36pLVOjmqhq6gemf9IbnNRObFqKILjyOQv8aU8pZBANLk1FrBWO9aWHMyR+1ArBrMeszDi+A1VaYArllJFNDtuxSfii0H1jHVfT2rfs3rYMnJT+aKKI0niTigWbtiLu4QzQ8jK7YSZ5ZmmeQDqoZpBvT+VcttdaHh63qcFr3BnzgHt5v3aiEZ5ihzl/tQ6dJ6INvsxmwswlc68kKaeOGPZ48WREdkoJaZTeYifz4AGepsYBSBN0zEDMy3UFLltwyxUNn2OgJ6xVkVSFEMkt9fkMGW+jrM0F/UuSuI5YGuQGxuulaQ5W/fdOUTg59qBB+QpjMcdRSslieC0acojFLYLEd4WId06CsbKJ1tCce09VVf9TqtdjVAKj1d0Dd13C/QjbsWX28HVRuNSOouRwf60ddlydzPH5zPWodsSMkceoRTTcMpDekrUtsPPn5r/+GvaNzGW5YP4xGS6yy1il/HVAnxeUPhcdCG5KVlVU9SFEPPfzVH2S6oDxIplGyt/nAQjkte9AbDAaTdaaoLqgcVX3p66nXDSSmqfAhAKYHmC1NidIsP2jwqJbpvy2ju3Kyon5mAibuiO5EpJeYVk5HJy5jHg3xD52ZoGMKK1y0m7VOGjCsbQ7UyeD4KeuG8nHlk7orCCLgoA54sxOIab3Rpbz9nVq2shFTHtxyRruNX4dXQM+sx4CY2eoVkCwWahDxUpxSduxbdFsKsb4hmf8BHh2p9yFT4xFFrAdI0rx0s2qpw6ikWZAZYTC7r5FCe201oAMNlmRY+6Z2bmNv7+k7TO1Ga3MSutcdZslDD7TYHE/usVXleMGD/sO+y/6F9BA5uNV7DV+AS9JV+DjuARZK7R+xChne7N2buzGb99mqSPAfIAUzhameahKxVfU0jW4wQFQybJZ4rqgNd9J3B0eP2BrfKe7Sb3JmQhwyzU149gfBYnLNnwIuDtea+CEma6le5G21Myk7KjhU0BVCZJ+VRbmHWYVlKEhkPO7SWqEXddJzxayjDOPOvQfOkTudtpsWajqdlh/vaWMpP47xoLbA6/Kp9vqqC1LeDS+HyGKDI3ivxnF8DCOPvVkCLpGFBMxWirBvmH7u8wn8XVU7GVSJ7GZvGoYpUqJmEmTVSgOkeLWnmHm+a2eeduA2ODVnaqssUnk3YkFRrtiiy8T6mieYLOphONGeZ2JkeIHrzTBVMhBZGoMjTE8TH6zK7dEHxO1oXIB+GYYzX7yAp6asAYU+h5ELdtR0CQ+SLhr3eLbBSkC35x0gX9sFJdhkkhqKk+Hbpve2y+OLUqMCGJJnAzRJowitqYEGp1vVwdFa2kRv2IrIuYBVepsDUk2qk6TEXQ1Fyhc9v+KVBqHf18ZC5FWFJOVNPp7US33R3FdpryDK0m1Ib2ywGet0/sJOBMhYF+mrYQGcwfSQBllvu32kt4HOMb4D/I2odebCuQC+V9htGL1e8u0tNy5/7bNmQtjYqDZuye8TQUkzpYJaaKGdjSlkYM8mBvnEhw+V6Shrc8nzSl46R8HwNZJLYhK9jIzTM0Z5EniqBbf38QABsveI/sLokmQEyrhRMHxIRobZHj1ZuaEDfmmQWkjUFpYMSb7xEvBfXNds4Syw3Ewb4z2uWJSOd+shvQ0vTeGLhe25IwO0qlHb94UmbwAKyGb0ru+x+qB/CNbi9meVbflKRx8x+qcOKlMyeK23u5/Kw8N+eDT+08fME7EbVsDpvp/S6V0zccCcHh12MBRgS1f6cNDrsHz+qH/rCCjDec8nidnwvtbGNNSZWWNDW5XwgUcjSGFgchmQoUlkODK+kwcdPv84KWfuhcyksieUUdveIYW20hvuaM9DH+TmyHj+wRoyzhYCFP8S7OOLABdEziDxfHS7ebwI4zv2gXtDI+OdRwexMuAWymrATHA63UAnOOqnG4w79aAD/FeUPOdnYkanAw+fWeyFPEyJs/oQ0ilQsKv8z3+bWHecBZUBGA3weNCVLW+O2hsh6CMXsIiv6XCIWsIXlXwO5OzrCFg+8vy7DIewRakFI+Pty69PjfyU/+Cwt3lQmhRPHQc0JWDFNN5gTQjanMeSSuDx+C7Hyi+xypyahx84HmbCHexGztzUB7L1LU1JdoLYbkDTczmSKn62TLs/2DLtfAXFtGlJzmiyKCmsX7c/Ok+xuUNguV+xQ3ngar2zV7CASHByjWv93Qm4rQspQhfP4jAi0X8rEsbyMFGHjpjQaRNCC1qi8oU8dUJvsM0ubKC+xG8KOHkH+Ihu6vCN8x0OUXAfTe9NY9aCGDARqGonyCHGnXguL5LAzLfeLOeuWvbVBoqS+H+LfrRJx5owBQ9zIOTAT2VJEjpgKwfbfOBwa/a8VnvSjzK200bZEDAeJ5gp/wRHgUdIsCoXU1XHVOZxucpY5w5iXlVGOaBKKcxcZFjnahI6ADfEaOU1kw3aLGOyYMpdgJtTP0wB1MyLOSZeTOhugrkLbevxgZzVLYhQp2TkAj1VFqQpd35zsaZX3s1txc7dNZ3SqM+EKvGENWcQA5GHIluhAx66YClkPv8SZarxcv++qgFJW5TiE3Y3OGzfjeBPfUqRB13v34HYpXCS6HzngwUWmz0L9O9xh/WtB/JM8Uge7FToXYg0DpO7je0N1mjAWek6YXfQdaXTXlsWXdUXo+lTcueGemK39FJDBMx6YAEqetbDHBUFJtD3D8Bq5e7dsPEUhogf3QEJdLarLJXGG85DyyNf714C86LRoQ+V7SKVm0R0cydq/m9oUGhnYQj+rjwm2UsPF+tv5Mdl+AS8CWnC0Xt2+vqW5aN4cLomtlUF6FxyoxOZD/8MaIXq0t5ZOmjJoKTLUBaFJl9Bng0/7g9AOPSPe6zX68kae4BAcNHBWcQU5LUDKruKCuqI6i4sACQdyQM5fb/Uz6MSzWNhF5erCtZf1iHNib6XlQItJstLD9mxKpHd0aPxqcza9nWCgdVsm7GBiS5IuVjDGv/FItV43EQuGWZZqWUc4/Kdjw/P23ddMLXi+hiRGV21bx+oOvFYkzqqmFcNomQO/kV6JieoV1Q4EnxDxEhZFQBihmotv1jLUFVsIVUYZsNTpAWr34BXHsj0F/k6+Pxj6rlbJfwXuCD52eZtQvxfZh7MEuOt7m2EpNYIM8G2QdTrI2tn3U1M/1aWyzXDiHxbvyNr+xGZ38lbK50T3kpMutaXavrm7OhL6UkR1LeYwOSAfMbTmMOSgMWyhbHcbGbX9FMDgqWhi0zME1Ypsbt362noGkNTPf/cVC5q+qtJqtK8n/8xoEwrgnAbL+rT49tWmlYLTcuQYSL1ZoiTVBkSsopOHi5qLrFmlAJUMxGg1e99gnn5sy1dkJDbnpFcLsBP+cgyX789Hj483wgXq6ddDW9zFJ7Lev2JiZVeKLCdjMbka50XNvIHAW02MnklnPbGS2TUsVoOdK1new2zpw4mHHqfaP0l3pLFFrzNwHEhAC/RaImvtXv9HjwQlgWhrKjKsdKYH6qynDqT/Q6cIssJUphJ+ebHGIikWmvyRIW4VIy9Eegc80/lYF94vjxRZjthFqRgIWMJdViBguj/9euz0vmjXZB+usQaNCgGh3u/ggimmgcV4/aQkrE6rKpbMZuV7Nz+F6rYM/rVDPPnf/df+z08GU+F0W/hb1nFojK6PmYX3j9aG19fj+9wUMQJ6PMtfVB1jCp4/XMY8t9qHwPdB/5oxl3QoMbzlMw90BVoJ+3iKO2yfrJ6EDT+NltMcAaDw6McA+/EwtNVA4RD9eNGtwSb8gpFWzXSqS94imW8tjGtKjJSRTD+5gh4kVisRvlNxznUF9CEPCBZSQzj1Fvgy/IjFejd/kOAS380fMxC1WrUQ2meeGzhobRIY9Ta5k9/3/51liKvf1I3ZI9wPA9QkvR7NL6jo15uxuJGlf7wzs5Ylb7eFlUVdlKfqgpLBYNH6idmYFjqtaGpviAr5acy+fMJdzOevglhzczy8d07mUwTW5/WvkXA0mOMtoKOMyiyaxRa7kXp2PDmnnaO99XjS7dC/kXm0a3Q19TtGR3zwEJWgPxbP5dHpcuBxdu+uKvG28bU66P/rYhZJut6Y5rZAZM54zuYDXExsg1z+YKdTZE4AlbG22FP80F5ExPThpqUNv7kUJbURrmhQPD+vrJJ9/ctBnjB4vKxtxAxFTHZ0QWU5VQ6aqOuFBS4xhKkVN+iHHJRP9e0qx2Fm7jKeHrQoGqqe7a4AQ4GfLLD3m0/L/GCOzVFUZjN4mJhY90aidhX7998Y66VmTz9mH3+kXEwHoUjQBuUyj7uYIkNyjVN5JjykOK2UeWhjh0H1VgtZqcRNlXJVCPNz3luHimObMNAGzoHTlsTzYrlnwHL8Bh/MsEE7pezwATdr/L6JywBwSlVwU5U1ihiNhNbg++uq7OSr8tBv7Gf/q7qZv/0Tzspph0NhCX97pnEDY3BXJfrxTbywy1G2tIGN7YB0Evt4W4H0xSEV2eIsDjt2fcgYOUhmtt2ChY68E7kUT1iVRJ8i3CTxoI321xDfWqrWdZ/CCfKcEEr0DQMVZJP7gVNscCzf23L8K88uFKr0wUTBCuhw1BBaDsf21W9bH2kU7fGqGNa6kp2YDlzHsyEWe60w/S/4+aifMDJ5+0NgHy+mLicBUMG6BnprKUCZKnuWrUcGWo8OnGE7Noh4dpRkrAjZU9H0WJHUlKn4NyO0kV0vLpAgwlrJEsKm0Ux2kY+Hf+JKtQWztqvWqa2arX/2rVqy7uSqlatrkpbXr/zrZgEKi+WTy6mXiOc+DkeGQucLAZL3LkuYlF7jcGNoiEdY0TXA8vPiZhs/P19qmotwdcEvJVkDh4UNsvVz9ToxoqqyqO7y7jYHXomElf9FuT+SzorelPSSFF6WZx1VPFTeFY6iV9qLLl1yMxSwZlOUY2grQGMZzh6ihCVOB8eWrIAg1nULNGfFLMsPpiUuqZCSwB5olmY3JaO9jHO6RV83pGG/YaFKR38L8HOSxvowZA9XT6RuwhlhY0vRwDBALcpEyZIbQ663L6NUDEDjBSCzptRMhpB7f3y3xCgGs/xnv3u+dvfPn9brac/BelkY2q6iX+Kg8RPPfxdH6zpTnX0cZvZEb7LCRiaXimVe5tkCcUrwZqnbUzfm8Rr9d+T0LkQRQ1d9XttXoBJb7gLdikmZasN4zR6rFRK96Kh8I6LteOwgki1ftzaueUcKFBAmIias7T5mHlx2lLVSEnrozLSLAgEJiAUg8NyFc0DSy1V+z3VY0BURIiKmPQjYqrDCF9fsaNeu5ZBK3FoyX+km5HsMO3EmnhYiMjoWfQ/VLgNpq6u7t6EnO/eUZWyddiVSk0aAoxfkZbPs8CZ23KZaXqjBw/v64nRREdr9JdzELwrTkcUfCWBmnQgKqaiqKPS1NRj6kviU/auyiXgAT+xKX8Jg+EiSG1Z1tTGBOr8s/DSFogD9UBnufWbYqYUWheuTYepRuN6OZbyZogSGJqqxpolqF4zsv66KCCedfIfxnbzkyNXovLbxqUmxpb6R/BZuV50tXwGiKiZoMS67kf6+x3uEF1tyf0y5mkaJcODgxmQbDaxnHBRGkl5UAdgcAieiOQAz6Qk6QHGCFDwHmx2+Y3S911AVHbV5Qv3/hFKUEqwrxWrgrk6c5DdOMsO64UPjo8r0hUeV0kV5BD+WnYCktmlMvtZglnMHA/Ka/rtVwv76ebVokr4ZK3GeRb7CZ8K86T9/5IByjTPeJbORybOqpOPsyHBAJvZCyADLPHcdHpgd175RewhFSN5xtJdGhwvJkadaVB7VDgGEA239IMTa3JR8kmVg8qlEZ1qFZkzKrQ1JgIq5L3R7QZhF3+sTRabko+g2118/SkxCfBIf/CA1q0/XOFS3ewSKpdVXkblEb4+e442Krgfcfn5u/fPAKmlajGlXyDAYjpFmcxCz9CP3VQLaKbsMbvfqxU3lAewsSdVdsZCMvCr9ZbIAMWaqgJMQYyKmHEuHsa8+6nXPemef/WDBctTrMYPKDIwsopdyB9zqNeeW6wrIrngCwuP60RmQ4QAbBJe1NneVLham3xj+tEJVM82qufSPM+31LWuEqWilXOFp11qo8iy3MbTHBm1CigofHMrA6e8VjxXQfg+0b9mJAI8jQG22QSs37D0Uz10viOW6fmhZZV/4oKkTN1WahZIa0KJZI0UBM2iZrPkaCT9u+vSghxoYDQilLwNS7QmN5RagNZS8hcD3/u/jDqj3A==", "estudio.html": "eNq1XFtv20iWfvevYJTuFplQ1M1ybMl0JnGcy8CZZOL09ACGkZTIksSYFzVJybIVAfML9mWB3X1bzMMusA8DLLAP87YPk38yv2B/wn6nqniTZHe6e7eBbpFVp06dOvdziu7De27kpNdTrk3SwD/aOaQfzWfh2K7xpHZ0OOHMPToMeMo0Z8LihKd2bZaOGvs1AIvhkAXcrs09fjWN4rSmOVGY8hBgV56bTmyXzz2HN8SL6YVe6jG/kTjM53abcKRe6vOjkySduV6kdZ8dNuXIzmGSXtNvP46idNloDMf9++1Ou9veHzQaUxZyH+/DttvZxbvvhbx/v8M6bncPr+ki7d/n+/yAc7wFM7wdsAOXMbwxx+nf7w17w1ELb9ElsIwO3L3haufBchgtGol344Xj/jCKXR43MLIinpjDyL1eBiwee2G/NZhwbzxJ++1W69vBkDmX4ziahW5/zmKdKDUGTuRHsXoHNcZgBLb027vThZZcJykPGjPPbLDp1OcNOWCe8XHEte9fmQkLk0bCY280iOY8HvnRVX/iuS4PVzv302i6nEYJ+BiFfTZMIn+W8gFGQcx0MfD5KJVPsaIQj66XTH123R/5fDEYs2l/H4PM98Zhw8POSd+BwHg8uGl4ocsX/d6AIBtXMUDpP6sdK+HjZQXNxqmFSIyB5Fu/TQeNfM/V5CwJKJtsxMz1Zomkbcpcl/jdnUrSOtOF3E4bztI0Ck1rmIbL0m5pDPZMWQySK1yGlPPdW2s7PSpthGetjV2kRLxwAkYD0yxOgGoaeYITVxMwpoFtHN4Po4IHiigrqpB0v+N0Rl22IfTKmj7Yx4Y+d5cR0Hrpdd/q9rJ9XT5iMz+lBdMl8bffxrPrFqKOuc9Sb87F8NE6U75eBFtodF3NCng4ywUcRiEfbCqZ1KiWVLb2Ns3/lTqwh+fAC6Wz6LcPaC5gi4aytr3WfFJYBJulkSDeiqY8XDvB0I+cS8zSoMZM+SvlUIUZqL3IkFO+SBvCLPpkRps2LABc7kQxE5yRfFLE75NeEcXVA9KZSnwSS3ItvUMFc+L7Ezpy5QhyaIsGZouSgPn+csM6wDeW9oUY4Un4YnqLHkWXuZjKOGiYKG5cZfJoAc0wAknBFp8kJ/rC1oRb6hGPyXpHURxIO4ZO8z/qDcwYJe9TcTTCFblezB2BHgTNgnCb98ocG2mMlOrB3vwKFLLQC5LlhhMkyVT93ODTLEm90XVDxTCFOUORKdCvtruSq8rUpLNmBz/potaIsrbRhVC3VZBiXA7cH41GQOWFo2iLCGU0QdASNr9HNG4qFWkEgiaXks6kuDugc2em27Z62CVJ2ZgPmTvmS8LXI2kpALLmkuQetYTkfD7mobu8zRWJDZWaPVpzFru9dWdRNkOVOTjOL/ZVubmvnb/iQ3Nm5Gex4N8015tXfG1x0g0Zy/HKiooCb7EDlaSAGK21yhB4y8zRFTzf4q26pRN4oZCgcqX34Tu2ScILkRPClVWo20LVdtPaVKeMZd3VjsPCOUvWffpvAu56TNMLZdnbxUGM5Y4mkqOMM6Rh+5nr2c8Ton3KL7SNBKNsegdVqXbFCmkihFTqktS67RuUI4k6Zwlfu8CnDeM1RdAyf6o26LYkdBpFfrKsurZcvRsLFQ21qqPKDtXboqvqVGsmWT5bu9eqHKqq6LRYKW0FvnrQskV29mCRxQl2xQmyVOigs9pZ7Rw2VdJ/mDixN001Kk3smhdQaRGwae1oqV6SWn9ZSycx57V+bZKm06TfbDpuaH1KXO5789gKedoMp0FTAP2mZbX3WlarOZx5vivHrCByZz7HipopUTXBryhMmj8PJV+wAGl80vyUBM3aaoVDCOJxiqYsn6h2wBuMWPNcuwZGo/LRNDHg+CxJ7Br0sSYmqZBKxDQApBw1l6WsMcdCvqhlC6KwdvQeajaL2WFTwh2twyNLBqOO3nr8hiU51FbUMqQTME+i5FaM2Py6dvQkdjzfv33fiQcYP70d4McZc7HXMX7iqELZYRNM2WAO+LV1wnUl03wivpZvo2Zh17Wjv//Ln7TT2Y3293/674KcEgrKl7ax26cCWBSltaw63X4cAlzAxL0oBqR60nS4R+MOlmMVfuB6HY5lT7LHW7dwYxYg6XRY7egZHr/8mZ5vhYZDCQH5ln40PUYJnrBYS6W6FGRJnt7JWYcFwXbG/s+//sO/acdf/oxww342dx271jJbtaPnVMbx7ecATHtfQJ2gEPNddivYgYA6ZW6keTc/WnewHcCNEjQin3Ur1m7PbO/XjrrN3VtBWuZ+i8wh9oa3UzdCEQldZ2VD/WnGIzvfzveTBfk/iPNOnmc4pAAq+2U/mTuiwC7oG7MQPIFX5H709z/9+zpcESpqmnDTIPZrCsZKWtirZGGqDVOkopsp8JY0uRrFBqJpM4E4r/otraW1KTNTmPf3c8dRHFcErc1xisebozK2Fe6axkSMvcUj5X5cBOw11afxOEorPvwJIm4Dg2zT9oX9pXAPxzCSuASwgfQK5RHMPfK//GUchVv8t5DfJffJln6ccZ+n0XaYCciDaf/jX7VjNq0El8I5Zz+VGC2DKQ4s47P2QGOJ9v7lu5MTbRRHgVYXQbM+yOaXb+Khlx5DlDEYtSoDZWHYUZPNCijCcQnLi9P3z0+hwEiWt6HwxVTSLMCqy5+9e3L85ivWl+CqCN5FUXASzr04CmFq6VYkvJhPmmsLJLYdHDVJtW/sxD5yI2cmZiCn+PoMsnLSKNYTY6Cgnr86PTmzl/Dm/fO6tNYPV3xojf1h3cwGSEdhEtVBGQC8m2ziwhQZAtBMRYqgBlUuQMOUDKhRCv0YYjL2q8GJRzQwDHxgCPhqVIT3gjjxKqdW2SF+9+r4xF6uk9+vvzh9qiGa8zjSMKjpCHZOpD3Ufvv25IWx/XhykRrRdCwztReRG6Wm9l7keZ+S0kr4JGs0XGDV86d/pGa1xjOL0PTvQySjpvZ9GHPmm9prD6ldhMU7mlpuDX04Dyx+Sr8ccV6OGLczWZKXeKWNCmCkGBTVIysafgLkm6e/BVQqontU2vYD1C3mCZxr17WS1Afo2ftTTY1++a9Q6z5DRRSUyPAjt/2h11IknL55lrG11/q2AtX50O6VoILZdQbZ7n0raCgpR78uk0kN2q9NZlAP4Cq0pF9/zaYMYVUTY5grqwuIhqPXqPvDiISq2tBaDGg0oOFYCCGOOJpOvIu5i9e4dMBCq7KVHIGSMkpwPsEZgqknkh+k29h2/OUvDHqkciBtEsUhcnNGCOeey6MPyHS8kMVWMN0Fxj/QmFQqQUUONobldpH3l8Hw/rf/FBAQ54cp5VvMmoZjALwKEDRD4oiY1ORkfZXbfCz0iMd2yK+kv7R+4MMXp+/UuL5kId2ZeCzpp/GMmyRyHs85LOMKkfLpbDTisZhawUVk6FDgpm+9BfffUatSf83SCbUedHkdU8yYHcMYlBedIa7qXhjy+Ad1ZYPHl8IjlNFHs3Q6g09G4D6jLrktST979+JpMVggRgDhUI0pCFaQT45Pzp57fuA574u5EiUioGPY4qHomtt0wsFO7h2pprKwCguOJ6jp9HypCwfrcwLKXWaCKpyXOHxG77oxEONWkZGUQMQp9NZCXjnlmKZBzIMS2NvX705evwCWmJGXzoig9hqBw/WjSLPFKotCg9yZ1q8FAt0wUUzuGpbUUJ5tCLUsb4dgNKVO6JwfswCb6shSC2k1S9IidG2z02rltDupL1BVwqmODcytvAO4Yv4zFLgkOSECGqY2wTvkLCnfNnY25ZBX2+pkG49jr8zaF3h9yf0plLvdMneRji86vNuWD53ObsfJJIMAqtPi/AgjP4rKpvKaJxO9xB4UPPwFjwKextd6p4XjG2ZJ7EqpqEZjvl6+iFkZoj0exRblYpTGWgu7Iczm7atmJ5tDjQbWSzzy6AWhAiSndMIDr0ToS7wm0wmYfErCgWKNxD84chf/7O6a1l751LS8UN9ZWEL1LOuFM38NVwcYZqHlsCStkIgxaU9WwKZk4GToYM/uvkn/Ke8L2Hzb2Avu3nZ/nzFs2y4jwCIg2EGYgzOGqGa+bwZwN+qRCRSJvVwhoYicy7LF0TvMAPl8afRptOhi8CX0ad+kFFZqjsRGOTuhYnPmSd0+w8HyA8jsTYwXCaCe2aZLqYWYLGV3mBXjxKFn3EGoid9CCfT6L20I+d4waQqUzbGfjpp1YyDJEjuUNhYwxLrRLBRM0kRXQ7+kzqYmST61l6op0T8vpG6ed8xds3thFhKBNpltq2WS/C/MrD+hFnVc1+xaWNYzoQC0bDhy97HM6pnWAS2Vy6jMyJsUci1r7cGoAHHe2MXGYvFe61GL9gRGGJL4V67PexYlYrvY47xr9sxGm9aORr3eo0cmydYC0e2eOWJ+grWig1FaZ+2a5y0s68lV0nSwbk/sKxetzi8vBiBa2AC5b2LxS77QT89bF9I0qK0eoky9tk/P2xdiKCtbhU1YlnV63rkwrGDmp97Uvz5zmM9i/SWCpBesY+1eyOEy1l1BAo1WEDc6D16a9G+j+wDIyLwry3oXyoZKxQKGH108liGkT/q+CfKqhGJP7OyN9Hu0zlhu2eOhtUdt1lzDhrA13aBbB2s2deEX4fBib/FDFCOgkhDpCmQBr3HJT4Jpep3bznDDQmlvwoNCdY4gxfXIPlqCmMjyEvLSRvZydklBypVjsLNLkRKHigAgisDmAMkFf0ohmVIcInIwxPD0GiiGxbBBGYB/LanexVxQ0G+sRJnMoRnYZqyCwk/gzsG+ehPizywEM/WhsVpRLoXoDc5GixKfR7AErrMbG7bBfbvdETYtua846tiEaczTY3FhUApqfxAFIFhsmEkGJLK0bSCDl7bM9dhCT6xrEzmAFI10eeXJhUkAiXVjPGhbB00xk7KQEgJrFM0fZMEP2a2BCJgjYTa7ySepQ8dt7pcHBChbMwDHWjwUMMjmdWZIeCdKdG48cE3Hui5m1cjNwxxmHV4eCUkHvBO4IaVHd5rYFWl9bLtNJD7idURvD3ryRerY2zj6JEOZlKkuc51cAcsWQvVZAhOheJbaLXNutwZ3qfl3392LrBkS9GfwnBb1ZWBeMgmyC/UazB/aY4ul2H0IZUwKVjnQu3SQ0rRomD1Wv3Ki2e3fsazZJQXM5fQVygLYb/S6uIqqG5ZMHt+/PrU/frMU/BZJsZ4iKY1OI/pm6wxbh2PdWGl4+PLncDzzUWv97a/aN8v5FqD5l/+IEQB4cjiMj75ZQucA9BzZgKt3MP3lnzUavN42eFMZDD4+JGvWiVJL1J3J449y39LQSpakycd+vQ7zZ8l16Gi5KPkCFViij4wlJKCEEtvsinnIK3nqTPSRuYR4JpHbr788efKsXpgzCp7LlcMICAJVgyLqrFY76xslk+hKp3ucUuR2WOgmtmjdnNPUxYA0auT5XGQyAxTFukpxtWgkwQ3olchsrAkjuo2lgB8NhjFnlyvl7GnMkBRl8qQEA/IUTVpL9WjtOt3M1gv1GKujq2yEfp7QOXRCCJsWas0hhZT5Rgkr1SfH6hu/j5tdY62iOu1W6wEXuLnbzJCtvv0o9RQbwJaMpQxrKJGiORcjK8wUud4aQGlisJYQrmQQgvGIFYO7A1K0kSlH2zL8CIXbDFlXcIwdUNgIsQ9EcAE6EqZt23Vqj9U/f85eJ17dEJFClBprFcsZ3KyLvKqoRGQHHInYwdBx9kwwbzwJeZL0rV7PpM8sffnW6knOVbZW/bo6neirdwRPIElR5arGQmnTRyvEMrlTkdmTYAbrYSsIhZcJvNC6FuzOXdJ1ww5CVTcVg3CfVNBVR4C0KBEGRYkwILUT3faKd6rXlfLAF1KDRoBbPg/HKaSqMOXHf5KBvKaJ/BSZGYjrBfu8/o6DJOpmvUTFy2I8vPZi8fskgap79PQ7PhYjZ1FMXRhqYj33qXWPhxcK+h0fqpFj2VCipyhGcU0LmT9zmXqSUE9h33I7FoqO2dso4R/e1y8kkZUjJlGc6jozh4Z9pAvCZXB4Az9h0Xe4xsODA+Pbg4PG2uywMqvOX7gch1xOeSdjqYRw7oiVF7bgquX43vSJmKFoK7BkfBzaeWvGgXtKueoj6HV5f1CnLKvsOyRmGBy8k8P1+gecXRNQUYh9UBrqOCa5Ll2CGhV1KLd+hirXE8AZ5ZlELx5nT/3yEVESKKzyaCr6U9NEYCP3DLd7BgejLP42Z1JOaLMVqh1HBJOLAr3Z15/2PQVTnhT3QacIT5mPqwuGkfnx9AckEToBUzZRAaYLEuaFiQTPw7m84doIAOvuonBW4hag/rguvrCp9+t0nVcf4HhFZ+lwb7dlVJAXZJBzqEPX65sUFEa7vvvj+uEw/9jgiPKDY+YyTThCzdZmoYzlFs08i6DHYIUW8BtsG5qazzTRPqZBN4LBWvW+UIH1I9EmsmON+ITYwONppG5Oj+oPBdSpIFY3+tKvyFwdVbqoviH0U9n7VJng4PYAK9kGFNuvcJ74PpaK7zu0zCaQaMcnDFnF0D4alliaRuOxL9XAHFp0kwxFsOY4mcgrqjlq5RQqsUk4lz2j12xKzvoXpayB/SSO2TXgxK9ehBfjcfEMS+oXb8o1B999Fwj7AnaiRaQwcsQwxAAVBXLArN+vP8yK67EorrMU0lBRSGVc56jPafEFNbJ0/Tw0nQs4iY+H4nrby66lS1fM3yydFV3Vekg/w5W8ufxoWJ8i1BkiSVSNKooSMvyU+EruJMyYgbJHeZbwgmzjHsuyLpnGEIbvvlMP92ybGeoZJYjL38xS3eoaA2ZR+z7VDTH6KqRBS2wkhCTIYD+lQ+WPvL5Wh8qu17bDDb8LVpeqHuV1olCU7fcoaKrDSmZcJfb5xS1aVVWlK1Qps2SiRwYkiZeM2Cv76MqSn/FnTu/KuLWLcC9T1s+f1zcoUl+heBstBv1rdLh/XjxfFBwN7KNA7CKcgg12qIJglW+X68fVelu6KPfM6sxTlnjOZu63P9wfjkZmvp3Mykp/7iAHSk1r0+XTdPJD7KW8L4sRI4/KV1U2yVT2ypKd/jci6ekgtSXffUVWtsp7tsLf2XXVaKyX7lw2VTH7W4R1LcxCOJURHH4ymqLonrKxiL1FB9YeKhVQyYIaj2x3e5RDyJdB5m6SStQs7KPFtvAqMBnCjiPDXQ9mcppcT74Rxk/meCAguu7R6+KEdZOylP8jcow7eX0uv+66uI3XRMgyl18RNfz1ULb6mn2cn9inegskMyQKjXGU3pnQeJSNZqQ5FKrp+yjULj+7G6ZAU3tz0rEWpmxcLlCHvHxgtR5RQylL/Nd7UynAU+sa/948BHCvJxtC5eZSutYjKtyApOKcmciSSyezkBR4yH5NcIOC1e9mwZAqVtUKNPlPieHOTKEsC9FuKCUJpEaVJKwMfEsKl8UKlcUVgrxL6Peq71vFXw5CVXBjJfdx0nhtnywD2yklvVU6pJcI7XtfkRYPtsKU6SK3Xop4ii6Vmt+x8UbyvmXjDZj1jXfyMFLuZRRXlqo9ru5LRfVaAOYxyxKR4D18tjLFbTD0Ob/8o8xO+cav1E6R5nFn0+RXhtNlYJUimoxKgaVimi0uYkUwUo37r238DP6/ic4obA+qB1Bhl2jO9GayaTj5/bp80MWxTPihzImxW6tnVqekcYKcwN5ySw91ohD//btTve7RpyZN+uyEVrjRVUhlil135Id84oMUTAiydNLyzZBGXxPdcBnTluQmmfjawN7+hcHdLfWv/ahk/auVvF9zGkXI8Quzc1NbXBlTWHjG/ZTpqnZHqVEu4d1NX30b92nvHV20b8VGUDvqDYtrZdkd1cv94XrsjT+IfvOnRFTc4he8zDvDRUBAdZpfSL8ZEnOsOfNR0soeNoudiTePks+flyppW+sAb/l6buOTKfPXff5nlr/sKn/GVf1s68LArvILh22EqsNd8utEp6/6jDVwyUAoSOAl8DgIbVTF4VcWcap3Li1WwuaNekN2vyUuxaZfGjGL/otswpcC5oWVIE/XQUOl2166Sglssmz1OXX967iw0bz/GZY+KhswvRR9FEJ+Prp4+FE7FH/libp2lOUaFg49hdEY9DcoYvIj3FW5U8aEa12jXZBJ9N5TyH8dreVKc7R1f4n998I+4LrOOFnDW4awn+iwcKYSMxoVvKbCZG7/ngxfr8/rxufP9ZTuMohkdaVymwzndlkyAhiK4YWufmkfybWXt63FNqKrr2K02l+mEpS4tuvG1txClhAq31XfgaUZ+TxlU+IeXa6kGaOHQr+KD+w3mqU8xcBau0l261a5iclrJ2oNGZQ85X+GdNgUf3902JT/i4f/BaU93Xk="}""")
for _n, _d in _FILES.items():
    open(f"{APP}/ap3d/{_n}", "wb").write(zlib.decompress(base64.b64decode(_d)))
if APP not in sys.path:
    sys.path.insert(0, APP)
# Si ya habia una version cargada, se cambia por la nueva SIN perder los
# modelos que ya estan en la GPU (antes Colab seguia usando el codigo viejo).
_old = sys.modules.get("ap3d.engine")
_eng = getattr(_old, "_ENGINE", None)
_app = sys.modules.get("ap3d.app")
if _app is not None:
    for _k in ("inline", "web"):
        try:
            _app._SERVERS.pop(_k).close()
        except Exception:
            pass
    try:
        _app._SERVERS.pop("tunnel").terminate()
    except Exception:
        pass
for _m in [m for m in sys.modules if m == "ap3d" or m.startswith("ap3d.")]:
    del sys.modules[_m]
if _eng is not None:
    from ap3d import engine as _e
    _eng.__class__ = _e.Engine
    _eng._ensure()
    _e._ENGINE = _eng
    print("Código actualizado; los modelos siguen cargados en la GPU.")
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip() or "SIN GPU: cambia el entorno a GPU")
import psutil
print(f"RAM del sistema: {psutil.virtual_memory().total/2**30:.0f} GB · CPU: {os.cpu_count()}")
print("Proyecto listo en", APP)

## 2. Instalar dependencias, TripoSG y Blender 4.5 LTS

In [ ]:
# @title 2. Instalar dependencias, Hunyuan3D-2, TripoSG y Blender 4.5 LTS
USAR_CACHE_EN_DRIVE = False  # @param {type:"boolean"}
INSTALAR_TRIPOSG = True  # @param {type:"boolean"}
# @markdown TripoSG es el modelo abierto del equipo de **Tripo AI**: formas con más detalle y bordes limpios. Se suma como segundo motor; si no instala, sigue Hunyuan3D.
# @markdown Si lo activas, las piezas compiladas se guardan en Drive y la próxima sesión se instalan en segundos.
import os, subprocess, sys, glob, shutil, time
t0 = time.time()

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if check:
            raise RuntimeError("Fallo: " + cmd)
    return r.stdout

# --- sistema (Blender sin ventana necesita estas librerias)
sh("apt-get -qq update && apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 "
   "libxxf86vm1 libxfixes3 libgl1 libegl1 libglu1-mesa libopengl0 > /dev/null", check=False)

# --- Python
sh(f"{sys.executable} -m pip install -q 'gradio>=5.9,<6' 'diffusers==0.32.2' 'transformers==4.49.0' 'accelerate>=1.2,<1.6' "
   "einops omegaconf trimesh pymeshlab pygltflib xatlas rembg onnxruntime-gpu "
   "opencv-python-headless ninja pybind11 psutil hf_transfer 'huggingface_hub>=0.25'")
print(f"Paquetes listos ({time.time()-t0:.0f}s)")

# --- Hunyuan3D-2
if not os.path.exists("/content/Hunyuan3D-2"):
    sh("git clone -q --depth 1 https://github.com/Tencent/Hunyuan3D-2 /content/Hunyuan3D-2")
sh(f"cd /content/Hunyuan3D-2 && {sys.executable} -m pip install -q -e . --no-deps")

import torch
cap = "%d.%d" % torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "8.0"
os.environ["TORCH_CUDA_ARCH_LIST"] = cap            # compila solo para tu GPU: mas rapido
os.environ["MAX_JOBS"] = str(os.cpu_count())
tag = f"py{sys.version_info.major}{sys.version_info.minor}_torch{torch.__version__.split('+')[0]}_sm{cap}"
cache = None
if USAR_CACHE_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache = f"/content/drive/MyDrive/ap3d_cache/{tag}"
    os.makedirs(cache, exist_ok=True)

def have(mod):
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True).returncode == 0

if not have("custom_rasterizer"):
    whl = glob.glob(f"{cache}/custom_rasterizer*.whl") if cache else []
    if whl:
        sh(f"{sys.executable} -m pip install -q {whl[0]}")
        print("custom_rasterizer desde la cache de Drive")
    else:
        print("Compilando custom_rasterizer (3-5 min)...")
        d = "/content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer"
        sh(f"cd {d} && {sys.executable} -m pip wheel -q . --no-build-isolation --no-deps -w /content/whl")
        w = glob.glob("/content/whl/custom_rasterizer*.whl")[0]
        sh(f"{sys.executable} -m pip install -q {w}")
        if cache:
            shutil.copy(w, cache)

d = "/content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer"
if not glob.glob(f"{d}/mesh_processor*.so"):
    so = glob.glob(f"{cache}/mesh_processor*.so") if cache else []
    if so:
        shutil.copy(so[0], d)
    else:
        sh(f"cd {d} && c++ -O3 -Wall -shared -std=c++11 -fPIC "
           f"$({sys.executable} -m pybind11 --includes) mesh_processor.cpp "
           f"-o mesh_processor$({sys.executable}-config --extension-suffix 2>/dev/null || "
           f"python3-config --extension-suffix)", check=False)
        if not glob.glob(f"{d}/mesh_processor*.so"):
            sh(f"cd {d} && {sys.executable} setup.py build_ext --inplace", check=False)
        if cache:
            for f in glob.glob(f"{d}/mesh_processor*.so"):
                shutil.copy(f, cache)
print(f"Hunyuan3D-2 listo ({time.time()-t0:.0f}s)")

# --- Blender: 4.5 LTS (el mas nuevo estable); si no se puede, 4.2 LTS
if not os.path.exists("/content/blender/blender"):
    for ver, serie in (("4.5.3", "4.5"), ("4.5.2", "4.5"), ("4.5.1", "4.5"), ("4.2.3", "4.2")):
        url = f"https://download.blender.org/release/Blender{serie}/blender-{ver}-linux-x64.tar.xz"
        r = subprocess.run(f"wget -q {url} -O /content/blender.tar.xz && tar -xf /content/blender.tar.xz "
                           f"-C /content && mv /content/blender-{ver}-linux-x64 /content/blender",
                           shell=True)
        subprocess.run("rm -f /content/blender.tar.xz", shell=True)
        if r.returncode == 0 and os.path.exists("/content/blender/blender"):
            break

# --- TripoSG (modelo abierto del equipo de Tripo AI): forma de alta fidelidad
if INSTALAR_TRIPOSG and not os.path.exists("/content/TripoSG"):
    sh("git clone -q --depth 1 https://github.com/VAST-AI-Research/TripoSG /content/TripoSG", check=False)
    sh(f"{sys.executable} -m pip install -q peft jaxtyping typeguard scikit-image", check=False)
    r = subprocess.run(f"{sys.executable} -m pip install -q diso", shell=True,
                       capture_output=True, text=True)
    print("TripoSG:", "listo" if r.returncode == 0 else "diso no compilo (TripoSG desactivado)")
print(sh("/content/blender/blender -b --version | head -1").strip())
print(f"Instalación lista en {time.time()-t0:.0f}s")

## 3. Comprobar GPU, Blender y modelos de IA (carga todo en RAM y GPU)

In [ ]:
# @title 3. Comprobar GPU, descargar modelos, precargarlos en la RAM y cargarlos en la GPU
CARGAR_MULTIVISTA = True  # @param {type:"boolean"}
CARGAR_TEXTURA = True  # @param {type:"boolean"}
MODO_CALIDAD_TAMBIEN = True  # @param {type:"boolean"}
CARGAR_TRIPOSG = True  # @param {type:"boolean"}
CARGAR_TEXTO_A_3D = True  # @param {type:"boolean"}
# @markdown Turbo (5 pasos + FlashVDM) siempre se carga: la forma sale en segundos. Con la casilla de arriba también queda cargado el modelo de 50 pasos para el preajuste "Maxima".
import os, sys, time
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
sys.path.insert(0, "/content/ap3d_app")
sys.path.insert(0, "/content/Hunyuan3D-2")
# Hunyuan3D-2 necesita transformers 4.49: con versiones nuevas su codificador
# de imagen (DINOv2) no carga ("Missing key(s) ... main_image_encoder").
import importlib.metadata as _md, subprocess
if _md.version("transformers") != "4.49.0" or not _md.version("diffusers").startswith("0.32"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.49.0",
                    "diffusers==0.32.2", "accelerate>=1.2,<1.6"], check=True)
    if "transformers" in sys.modules:
        raise SystemExit("Versiones corregidas. Ve a Entorno de ejecución › Reiniciar sesión "
                         "y vuelve a ejecutar la celda 1 y después esta celda 3.")
import torch
from huggingface_hub import snapshot_download
from ap3d import config
from ap3d.engine import get_engine

print("CUDA:", torch.cuda.is_available())
P = config.detect()
print(config.describe(P))

t = time.time()
pats = ["hunyuan3d-dit-v2-0-turbo/*"] + (["hunyuan3d-dit-v2-0/*"] if MODO_CALIDAD_TAMBIEN else [])
if CARGAR_TEXTURA:
    pats += ["hunyuan3d-paint-v2-0-turbo/*", "hunyuan3d-paint-v2-0/*", "hunyuan3d-delight-v2-0/*"]
snapshot_download("tencent/Hunyuan3D-2", allow_patterns=pats, max_workers=16)
if CARGAR_MULTIVISTA:
    snapshot_download("tencent/Hunyuan3D-2mv", allow_patterns=["hunyuan3d-dit-v2-mv-turbo/*"]
                      + (["hunyuan3d-dit-v2-mv/*"] if MODO_CALIDAD_TAMBIEN else []), max_workers=16)
if CARGAR_TRIPOSG and os.path.exists("/content/TripoSG"):
    try:
        snapshot_download("VAST-AI/TripoSG", local_dir="/content/TripoSG/pretrained_weights/TripoSG", max_workers=16)
        snapshot_download("briaai/RMBG-1.4", local_dir="/content/TripoSG/pretrained_weights/RMBG-1.4", max_workers=16)
    except Exception as e:
        print("TripoSG no se pudo descargar:", e)
        CARGAR_TRIPOSG = False
if CARGAR_TEXTO_A_3D:
    try:
        snapshot_download("Tencent-Hunyuan/HunyuanDiT-v1.1-Diffusers-Distilled", max_workers=16)
    except Exception as e:
        print("Texto a 3D no disponible:", e)
print(f"Modelos descargados ({time.time()-t:.0f}s)")

eng = get_engine()
if P["preload_weights"]:
    eng.preload_weights()          # deja todos los pesos en la RAM
eng.load(want_mv=CARGAR_MULTIVISTA, want_paint=CARGAR_TEXTURA, want_turbo=True,
         want_quality=MODO_CALIDAD_TAMBIEN)
if CARGAR_TRIPOSG:
    eng.load_triposg()
print("Todo listo. Sigue con la celda 4 o 5.")

## 3b. Prueba rápida sin página web (opcional)

In [ ]:
# @title 3b. Prueba rápida SIN página web (confirma que el motor funciona y mide tiempos)
import glob, sys, time
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
_img = sorted(glob.glob("/content/Hunyuan3D-2/assets/example_images/*.png"))[0]
_t = time.time()
_r = get_engine().run(_img, steps=5, octree=384, faces=100000, tex=2048, rig="auto",
                      progress=lambda f, m: print(f"{int(f*100):3d}%  {m}", flush=True) if int(f*100) % 10 == 0 else None)
_i = _r["info"]
print(f"\nLISTO en {time.time()-_t:.0f}s  ->  {_r['zip']}")
print("forma", _i.get("segundos_forma"), "s · textura", _i.get("segundos_textura"),
      "s · Blender", _i.get("segundos_blender"), "s · avisos:", _i.get("avisos"))

## 4. Estudio

In [ ]:
# @title 4. Estudio (dentro del notebook)
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_inline
launch_inline(port=7860)

## 5. Abrir la página web con contraseña

In [ ]:
# @title 5. Abrir la página web con contraseña (para el celular)
CONTRASENA = ""  # @param {type:"string"}
# @markdown Déjala vacía para generar una al azar. El usuario es **ap**. La página deja de funcionar al cerrar la sesión de Colab.
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_web
url, user, pw = launch_web(CONTRASENA, user="ap", port=7861)
print("Página web:", url)
print("Usuario:", user, "· Contraseña:", pw)

## 6. Guardar en Drive

In [ ]:
# @title 6. Guardar todos los resultados en Google Drive
CARPETA = "AP_Model_3D"  # @param {type:"string"}
import os, shutil, glob
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
dst = f"/content/drive/MyDrive/{CARPETA}"
os.makedirs(dst, exist_ok=True)
n = 0
for z in glob.glob("/content/ap3d_salidas/*.zip"):
    if not os.path.exists(os.path.join(dst, os.path.basename(z))):
        shutil.copy(z, dst); n += 1
print(f"{n} modelos nuevos copiados a {dst}")

## 7. Memoria

In [ ]:
# @title 7. (Opcional) Liberar GPU / ver uso de memoria
LIBERAR_MODELOS = False  # @param {type:"boolean"}
import gc, torch, psutil, sys
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
e = get_engine()
if LIBERAR_MODELOS:
    e.shape = e.shape_mv = e.paint = None; e.ready = False
    gc.collect(); torch.cuda.empty_cache()
e.log_mem()